# STOFS-3D-Atlantic: CF and UGRID post-processing

This notebook post-processes STOFS-3D-Atlantic (v3.1) output files into CF-1.11 and UGRID-1.0 compliant copies, checks them, and tests that the data did not change. The original files are never modified.

## What to install

**Python packages** (the scripts and tests)

| Package | Used for |
| --- | --- |
| `xarray`, `numpy`, `netcdf4` | Reading and writing the netCDF files (all scripts) |
| `xugrid` | Opening the files as UGRID meshes (comparison test) |
| `dask` | Writes the 7 GB variables one time step at a time (about 600 MB). Strongly recommended: without it, writing a 3D file needs one free block of about 7 GB and can fail with a `MemoryError` |

**Checkers**

| Package | Command | Used for |
| --- | --- | --- |
| `compliance-checker` (IOOS, version 6.1.0 was used) | `compliance-checker --test=cf:1.11 FILE.nc` | CF check |
| `ugrid-checks` | `ugrid-checker FILE.nc` | UGRID conformance check |
| `cfchecker` (CEDA, optional) | `cfchecks -v 1.11 FILE.nc` | Second CF checker; needs the UDUNITS library |

**Example install** (a conda environment, then pip for the checkers):

```
conda create -n stofs-cf -c conda-forge python=3.12 xarray numpy netcdf4 xugrid dask
conda activate stofs-cf
pip install compliance-checker ugrid-checks
```



## How this notebook works

Each file type has its own section below, in the same order every time:

1. **What will change**: a table of the metadata changes for that file type.
2. **Make the compliant copy**: a cell that reads the original and writes a `*_compliant.nc` copy. The original is never modified.
3. **Check it**: run the two checkers in a terminal (commands below).
4. **Test it**: a cell that compares the new file with the original, variable by variable.

Set `src` and `out` in the fix cell, and `ORIG` and `NEW` in the test cell, to your own file names. The names in the cells are examples.

### The workflow

The fix is metadata (attributes) and a few storage types, not variable names. Every file is read undecoded (`mask_and_scale=False`, `decode_times=False`) so attributes and fill values are seen as stored, edited in memory with xarray, and written to a new file.

### Changes common to all files

| Item | Change | Reason |
| --- | --- | --- |
| Mesh variable `SCHISM_hgrid` | Rewritten as a dimensionless `int32` scalar; attributes kept | The UGRID mesh variable has no dimensions (`ugrid-checker` A101) |
| Mesh attributes | `edge_coordinates` and `face_coordinates` removed from the mesh variable; the face and edge centre variables stay in the file | Keeping them makes `ugrid-checker` ask for `standard_name` on the centres (A203), which conflicts with the IOOS CF checker's one-longitude, one-latitude check (next row) |
| Face connectivity | `int32`, `_FillValue = -1`, `start_index = 1`, 4 columns kept | About 2% of faces are quads; triangles are padded with the fill value |
| Edge connectivity | `int32`, no `_FillValue` | Every edge has exactly two nodes, so no entries are padded; a fill value triggers `ugrid-checker` A304 |
| Longitude and latitude | `standard_name` only on the node x and y; face and edge centres keep `long_name` and `units` | The IOOS CF checker (section 5.6) requires exactly one variable with `standard_name` longitude and one with latitude; the originals had three of each |
| Names and units | `long_name`, `standard_name` (only where CF has a fitting name) and UDUNITS-valid units | CF sections 3.1 and 3.3 |
| Data variables | `mesh = SCHISM_hgrid` and `location` (`node`, `face` or `edge`) | UGRID readers use them to link each variable to the mesh |
| `time` | `_FillValue` removed; `calendar = "standard"` added | CF forbids a fill value on coordinate variables |
| Global attributes | `Conventions = "CF-1.11 UGRID-1.0"` and `title` (`history` too for `field2d`) | CF section 2.6 |

### Variables shared by all 3D files

The seven 3D file types (zCoordinates, verticalVelocity, temperature, salinity, horizontalVelX, horizontalVelY, diffusivity) also contain these three variables, changed identically in each.

| Variable | Changes |
| --- | --- |
| `depth` | `standard_name = sea_floor_depth_below_geoid`, `long_name = bathymetry at nodes`, units `m`, `positive = down` |
| `minimum_depth` | `long_name = minimum water depth for wetting/drying`, units `m`; no standard name |
| `bottom_index_node` | `long_name = bottom layer index at each node`; no standard name |

### Checking a new file

Run these in a terminal (see the packages at the top):

```
ugrid-checker FILE_compliant.nc
compliance-checker --test=cf:1.11 FILE_compliant.nc
```

Expect `ugrid-checker` to report no problems, and the CF checker to report only the four section 9.5 `cf_role` messages. Those four are false positives: the checker only knows the `cf_role` values for sampling geometries (`timeseries_id`, `profile_id`, `trajectory_id`), while UGRID requires `mesh_topology`, `face_node_connectivity` and `edge_node_connectivity`.

### Fill and missing values

CF wants one fill attribute per variable, and `missing_value` must equal `_FillValue` when both exist. The rule used is: **`missing_value` is converted into `_FillValue`, never deleted.** An early version deleted it, which unmasked 1.42 billion fill values in a zCoordinates file; the test cells caught it.


## Inspect a file

Optional. These two cells print an overview of an original file: dimensions, every variable with its attributes, and the global attributes (including the NCO history). Use them to see what a new file type needs before writing its fix cell.


In [ ]:
import xarray as xr

filename = "stofs_3d_atl.t12z.fields.out2d_f001_012.nc"
ds = xr.open_dataset(filename)

# Print high-level dataset overview (dimensions, coordinates, variables)
print(ds)

# Inspect attributes on potential topology variables
for var_name in ds.data_vars:
    attrs = ds[var_name].attrs
    if attrs.get("cf_role") == "mesh_topology":
        print(f"\nMesh Topology Variable '{var_name}' attributes:")
        for key, val in attrs.items():
            print(f"  {key}: {val}")

In [ ]:
import xarray as xr

filename = "stofs_3d_atl.t12z.field2d_f025_036.nc"
report = filename.replace(".nc", "_variables.txt")

# Open without decoding so the attributes are shown as stored in the file
ds = xr.open_dataset(filename, mask_and_scale=False, decode_times=False)

lines = []
def out(s=""):
    print(s)
    lines.append(s)

# 1. Overview
out(str(ds))

# 2. Mesh topology variable (checks coordinates and data variables)
for name, v in ds.variables.items():
    if v.attrs.get("cf_role") == "mesh_topology":
        out(f"\nMesh Topology Variable '{name}' attributes:")
        for k, val in v.attrs.items():
            out(f"  {k}: {val}")

# 3. Compact table of every variable
out("\n=== Variable summary ===")
out(f"{'name':32s} {'dims':55s} {'dtype':9s} {'standard_name':32s} {'units'}")
for name, v in ds.variables.items():
    out(f"{name:32s} {str(v.dims):55s} {str(v.dtype):9s} "
        f"{v.attrs.get('standard_name', '-'):32s} {v.attrs.get('units', '-')}")

# 4. Full attributes of every variable
out("\n=== Attributes per variable ===")
for name, v in ds.variables.items():
    out(f"\n{name}  dims={v.dims} shape={v.shape} dtype={v.dtype}")
    for k, val in v.attrs.items():
        out(f"    {k}: {val}")
    if v.encoding.get("_FillValue") is not None:
        out(f"    (encoding _FillValue: {v.encoding['_FillValue']})")

# 5. Global attributes, including the full NCO history
out("\n=== Global attributes ===")
for k, val in ds.attrs.items():
    out(f"  {k}: {val}")

with open(report, "w") as f:
    f.write("\n".join(lines))
print("\nSaved to", report)

## 1. zCoordinates files

**What will change** (besides the common changes and the shared 3D variables above)

| Variable | Changes |
| --- | --- |
| `zCoordinates` | `long_name = vertical coordinate of each node and layer`, units `m`, `positive = up`; no standard name |
| `zCoordinates` | `missing_value` converted to `_FillValue`. The `-1e20` placeholder values are left as they are |

The fix cell stops with an error if the variable has two different fill values, so nothing can be lost silently.


In [4]:
import numpy as np
import xarray as xr

# ---- set your files here ---------------------------------------------------
src = "stofs_3d_atl.t12z.fields.zCoordinates_f013_024.nc"
out = "stofs_3d_atl.t12z.fields.zCoordinates_f013_024_compliant.nc"
title = "STOFS-3D-Atlantic zCoordinates, f013-f024"
# ----------------------------------------------------------------------------

# The data variable is about 7 GB. With dask installed it is written one time step at a
# time (about 600 MB), so the whole variable is never held in memory. Without dask,
# writing needs one free block of about 7 GB and can fail with a MemoryError.
try:
    import dask
    dask.config.set(scheduler="synchronous")   # one chunk at a time keeps memory low
    chunks = {"time": 1}
except ImportError:
    chunks = None
ds = xr.open_dataset(src, mask_and_scale=False, decode_times=False, chunks=chunks)

# UGRID: mesh variable as a dimensionless int32 scalar, without the optional
# face/edge coordinate references (they conflict with CF's one lon/lat rule)
mesh_attrs = dict(ds["SCHISM_hgrid"].attrs)
mesh_attrs.pop("edge_coordinates", None)
mesh_attrs.pop("face_coordinates", None)
ds = ds.drop_vars("SCHISM_hgrid")
ds["SCHISM_hgrid"] = xr.Variable((), np.int32(0), mesh_attrs)

# UGRID: connectivity arrays (int32; fill value only on faces)
ds["SCHISM_hgrid_face_nodes"] = ds["SCHISM_hgrid_face_nodes"].astype("int32")
ds["SCHISM_hgrid_edge_nodes"] = ds["SCHISM_hgrid_edge_nodes"].astype("int32")
for v in ["SCHISM_hgrid_face_nodes", "SCHISM_hgrid_edge_nodes"]:
    ds[v].attrs.pop("_FillValue", None)
    ds[v].encoding.pop("_FillValue", None)

# CF: standard_name longitude/latitude on node coordinates only
ds["SCHISM_hgrid_node_x"].attrs.update(standard_name="longitude", long_name="node longitude", units="degrees_east")
ds["SCHISM_hgrid_node_y"].attrs.update(standard_name="latitude", long_name="node latitude", units="degrees_north")
for v in ["SCHISM_hgrid_face_x", "SCHISM_hgrid_face_y", "SCHISM_hgrid_edge_x", "SCHISM_hgrid_edge_y"]:
    if v in ds.variables:
        ds[v].attrs.pop("standard_name", None)
        ds[v].attrs.setdefault("long_name", v.replace("SCHISM_hgrid_", "").replace("_", " ") + " coordinate")

# CF: names and units (check these against your data)
ds["zCoordinates"].attrs.update(long_name="vertical coordinate of each node and layer", units="m", positive="up")
ds["depth"].attrs.update(standard_name="sea_floor_depth_below_geoid", long_name="bathymetry at nodes", units="m", positive="down")
ds["minimum_depth"].attrs.update(long_name="minimum water depth for wetting/drying", units="m")
ds["bottom_index_node"].attrs.update(long_name="bottom layer index at each node")

# CF: fill values
ds["time"].attrs.pop("_FillValue", None)
ds["time"].encoding.pop("_FillValue", None)
# missing_value must equal _FillValue. Do NOT just delete missing_value: that
# unmasks the fill values (9.97e36) in the data. Convert it to _FillValue.
zc = ds["zCoordinates"].attrs
if "missing_value" in zc:
    if "_FillValue" in zc and zc["_FillValue"] != zc["missing_value"]:
        raise ValueError(f"zCoordinates has two different fill values: "
                         f"_FillValue={zc['_FillValue']}, missing_value={zc['missing_value']}")
    zc["_FillValue"] = zc.pop("missing_value")

# CF: time calendar stated explicitly (it is the default)
ds["time"].attrs["calendar"] = "standard"

# CF: global attributes
ds.attrs["Conventions"] = "CF-1.11 UGRID-1.0"
ds.attrs["title"] = title

ds.to_netcdf(out, encoding={
    "time": {"_FillValue": None},
    "SCHISM_hgrid_edge_nodes": {"_FillValue": None},
    "SCHISM_hgrid_face_nodes": {"dtype": "int32", "_FillValue": np.int32(-1)},
})
print("Wrote", out)

Wrote stofs_3d_atl.t12z.fields.zCoordinates_f013_024_compliant.nc


Open the result with xugrid (optional quick check):


In [5]:
import xugrid as xu
uds = xu.open_dataset("stofs_3d_atl.t12z.fields.zCoordinates_f013_024_compliant.nc")
g = uds.ugrid.grids[0]
print(g.n_node, g.n_face, g.n_edge)
uds

3052121 5872610 8924888


<xarray.Dataset> Size: 8GB
Dimensions:              (one: 1, nSCHISM_hgrid_node: 3052121,
                          nSCHISM_hgrid_face: 5872610,
                          nSCHISM_hgrid_edge: 8924888, time: 12,
                          nSCHISM_vgrid_layers: 49)
Coordinates:
  * nSCHISM_hgrid_node   (nSCHISM_hgrid_node) int64 24MB 0 1 ... 3052119 3052120
    SCHISM_hgrid_node_x  (nSCHISM_hgrid_node) float64 24MB ...
    SCHISM_hgrid_node_y  (nSCHISM_hgrid_node) float64 24MB ...
  * nSCHISM_hgrid_face   (nSCHISM_hgrid_face) int64 47MB 0 1 ... 5872608 5872609
  * nSCHISM_hgrid_edge   (nSCHISM_hgrid_edge) int64 71MB 0 1 ... 8924886 8924887
  * time                 (time) datetime64[ns] 96B 2026-09-30T01:00:00 ... 20...
Dimensions without coordinates: one, nSCHISM_vgrid_layers
Data variables:
    minimum_depth        (one) float64 8B ...
    depth                (nSCHISM_hgrid_node) float32 12MB ...
    bottom_index_node    (nSCHISM_hgrid_node) int32 12MB ...
    SCHISM_hgrid_face_x  (nSCHISM_hgrid_face) float64 47MB ...
    SCHISM_hgrid_face_y  (nSCHISM_hgrid_face) float64 47MB ...
    SCHISM_hgrid_edge_x  (nSCHISM_hgrid_edge) float64 71MB ...
    SCHISM_hgrid_edge_y  (nSCHISM_hgrid_edge) float64 71MB ...
    zCoordinates         (time, nSCHISM_hgrid_node, nSCHISM_vgrid_layers) float32 7GB ...
Attributes:
    NCO:          netCDF Operators version 5.0.6 (Homepage = http://nco.sf.ne...
    history:      Tue Sep 29 18:00:40 2026: ncatted -a units,zCoordinates,o,c...
    Conventions:  CF-1.11 UGRID-1.0
    title:        STOFS-3D-Atlantic zCoordinates, f013-f024

**Test.** Compares the new file with the original: values, statistics, fill handling, connectivity, and a read through xugrid.


In [1]:
"""
Check that a *_compliant.nc file still holds the same data as the original:

 1. every numeric variable has identical values (NaN positions included)
 2. statistics (valid count, masked count, sum, mean, min, max) are identical
 3. fill / missing values are understood: the number of masked (NaN) values in
    the decoded new file equals the number of raw _FillValue (+ NaN) values
 4. the connectivity arrays are unchanged
 5. xugrid opens the new file, builds the mesh with the right sizes and
    connectivity, and a xugrid reduction (mean) matches xarray on the original
"""
import sys

import numpy as np
import xarray as xr
import xugrid as xu

# ---- set your files here ---------------------------------------------------
ORIG = "stofs_3d_atl.t12z.fields.zCoordinates_f013_024.nc"
NEW = "stofs_3d_atl.t12z.fields.zCoordinates_f013_024_compliant.nc"

# variables that were renamed in the new file (field2d), else leave empty:
# RENAMES = {"uvel4.5": "uvel_4p5m", "vvel4.5": "vvel_4p5m",
#            "uvel0.5": "uvel_0p5m", "vvel0.5": "vvel_0p5m"}
RENAMES = {}

# which time steps to test: None = all (slow on 8 GB files), or e.g. [0, -1]
TIME_STEPS = None
# ----------------------------------------------------------------------------

SKIP = {"SCHISM_hgrid"}   # placeholder variable; its type was changed on purpose
FACE_CANDIDATES = ["SCHISM_hgrid_face_nodes", "element"]
EDGE_CANDIDATES = ["SCHISM_hgrid_edge_nodes"]

results = []


def check(name, ok, msg=""):
    results.append((name, bool(ok)))
    print(("PASS  " if ok else "FAIL  ") + name + (f"   {msg}" if msg else ""))


class Stats:
    def __init__(self):
        self.n = 0          # valid values
        self.nnan = 0       # masked (NaN) values
        self.sum = 0.0
        self.min = np.inf
        self.max = -np.inf

    def add(self, a):
        if a.dtype.kind == "f":
            nan = int(np.isnan(a).sum())
            self.nnan += nan
            valid = a.size - nan
            if valid:
                self.n += valid
                self.sum += float(np.nansum(a, dtype=np.float64))
                self.min = min(self.min, float(np.nanmin(a)))
                self.max = max(self.max, float(np.nanmax(a)))
        else:
            self.n += a.size
            self.sum += float(a.sum(dtype=np.float64))
            self.min = min(self.min, float(a.min()))
            self.max = max(self.max, float(a.max()))

    @property
    def mean(self):
        return self.sum / self.n if self.n else float("nan")

    def same_as(self, o):
        return (self.n, self.nnan, self.sum, self.min, self.max) == \
               (o.n, o.nnan, o.sum, o.min, o.max)


def step_indices(da):
    """Time steps to test for a variable (None = the whole variable at once)."""
    if "time" in da.dims and da.ndim > 1:
        n = da.sizes["time"]
        return list(range(n)) if TIME_STEPS is None else sorted({i % n for i in TIME_STEPS})
    return [None]


def get(da, i):
    return da.values if i is None else da.isel(time=i).values


def as_xr(x):
    return getattr(x, "obj", x)   # unwrap xugrid objects


# ---- open everything --------------------------------------------------------
orig_dec = xr.open_dataset(ORIG, decode_times=False)                       # fills masked to NaN
new_dec = xr.open_dataset(NEW, decode_times=False)
orig_raw = xr.open_dataset(ORIG, mask_and_scale=False, decode_times=False)  # as stored
new_raw = xr.open_dataset(NEW, mask_and_scale=False, decode_times=False)
uds = xu.open_dataset(NEW)

face_name = next(n for n in FACE_CANDIDATES if n in orig_raw.variables)
edge_name = next((n for n in EDGE_CANDIDATES if n in orig_raw.variables), None)
conn_names = {face_name} | ({edge_name} if edge_name else set())

# ---- 1-3. every numeric variable --------------------------------------------
print("\n=== Variables: values, statistics, fill/missing handling ===")
rows = []
for v in orig_dec.variables:
    if v in SKIP or v in conn_names:
        continue
    od = orig_dec[v]
    if od.dtype.kind not in "fiu":
        continue
    rn = RENAMES.get(v, v)
    if rn not in new_dec.variables:
        check(v, False, "missing in new file")
        continue
    nd, nr = new_dec[rn], new_raw[rn]
    if od.dims != nd.dims or od.shape != nd.shape:
        check(v, False, f"dims/shape differ: {od.dims}{od.shape} vs {nd.dims}{nd.shape}")
        continue

    so, sn = Stats(), Stats()
    identical = True
    n_fill_raw = n_nan_raw = 0
    fill = nr.attrs.get("_FillValue")
    for i in step_indices(od):
        a_o, a_n, raw = get(od, i), get(nd, i), get(nr, i)
        identical &= np.array_equal(a_o, a_n, equal_nan=(a_o.dtype.kind == "f"))
        so.add(a_o)
        sn.add(a_n)
        if fill is not None:
            n_fill_raw += int((raw == fill).sum())
        if raw.dtype.kind == "f":
            n_nan_raw += int(np.isnan(raw).sum())

    ok_vals = identical and so.same_as(sn)
    expected_masked = n_fill_raw + n_nan_raw
    ok_fill = sn.nnan == expected_masked
    ok_attr = ("missing_value" not in nr.attrs) or (nr.attrs.get("missing_value") == fill)
    check(f"{v}: values and statistics identical", ok_vals,
          "" if ok_vals else f"orig n={so.n} sum={so.sum} | new n={sn.n} sum={sn.sum}")
    check(f"{v}: fill/missing understood", ok_fill and ok_attr,
          f"masked in decoded new = {sn.nnan}, stored fill/NaN = {expected_masked}, "
          f"_FillValue = {fill}")

    # xugrid reading of the same variable (first tested time step)
    if rn != "time" and rn not in as_xr(uds).variables:
        # xugrid does not expose every variable (e.g. the "crs" grid-mapping
        # variable); the xarray checks above already cover it
        print(f"NOTE  {v}: not exposed by xugrid, xugrid comparison skipped")
    elif rn != "time":
        try:
            i0 = step_indices(od)[0]
            ua = uds[rn]
            xa = as_xr(ua)
            a_u = get(xa, i0)
            res_u = ua if i0 is None else ua.isel(time=i0)
            mean_u = float(as_xr(res_u.mean()))
            od0 = od if i0 is None else od.isel(time=i0)
            mean_o = float(od0.mean())
            ok_u = np.array_equal(a_u, get(od, i0), equal_nan=(a_u.dtype.kind == "f")) \
                and np.isclose(mean_u, mean_o, rtol=1e-6, atol=0, equal_nan=True)
            check(f"{v}: xugrid values and mean match original", ok_u,
                  f"xugrid mean = {mean_u:.6g}, original mean = {mean_o:.6g}")
        except Exception as e:  # report, don't stop
            check(f"{v}: xugrid values and mean match original", False, repr(e))

    rows.append((v, str(od.dtype), sn.n, sn.nnan, sn.mean, sn.min, sn.max))

# ---- 4. connectivity arrays (as stored) -------------------------------------
print("\n=== Connectivity arrays ===")
for name in sorted(conn_names):
    fo = np.nan_to_num(orig_raw[name].values.astype("float64"), nan=-1).astype("int64")
    fn = new_raw[name].values.astype("int64")
    check(f"{name}: unchanged as stored", np.array_equal(fo, fn))

# ---- 5. xugrid mesh ---------------------------------------------------------
print("\n=== xugrid mesh ===")
grid = uds.ugrid.grids[0]
mesh_var = next(n for n, x in new_raw.variables.items()
                if x.attrs.get("cf_role") == "mesh_topology")
node_x_name = new_raw[mesh_var].attrs["node_coordinates"].split()[0]
fn = new_raw[face_name].values.astype("int64")
start = int(new_raw[face_name].attrs.get("start_index", 0))
check("mesh variable present", True, f"'{mesh_var}'")
check("n_node matches", grid.n_node == new_raw[node_x_name].size,
      f"xugrid {grid.n_node}, file {new_raw[node_x_name].size}")
check("n_face matches", grid.n_face == fn.shape[0],
      f"xugrid {grid.n_face}, file {fn.shape[0]}")
expected = np.where(fn >= start, fn - start, -1)   # xugrid uses 0-based indices
gc = np.asarray(grid.face_node_connectivity)
check("face connectivity read correctly by xugrid",
      gc.shape == expected.shape and np.array_equal(gc, expected),
      f"shape {gc.shape} vs expected {expected.shape}")

# ---- summary ----------------------------------------------------------------
print("\n=== Statistics of the new file (decoded; fill values excluded) ===")
print(f"{'variable':28s}{'dtype':9s}{'valid':>14s}{'masked':>12s}{'mean':>14s}{'min':>14s}{'max':>14s}")
for r in rows:
    print(f"{r[0]:28s}{r[1]:9s}{r[2]:14d}{r[3]:12d}{r[4]:14.6g}{r[5]:14.6g}{r[6]:14.6g}")

failed = [n for n, ok in results if not ok]
print(f"\n{len(results) - len(failed)} passed, {len(failed)} failed")
for n in failed:
    print("  FAILED:", n)
sys.exit(1 if failed else 0)


=== Variables: values, statistics, fill/missing handling ===
PASS  minimum_depth: values and statistics identical
PASS  minimum_depth: fill/missing understood   masked in decoded new = 0, stored fill/NaN = 0, _FillValue = nan
PASS  minimum_depth: xugrid values and mean match original   xugrid mean = 0.0001, original mean = 0.0001
PASS  crs: values and statistics identical
PASS  crs: fill/missing understood   masked in decoded new = 0, stored fill/NaN = 0, _FillValue = None
NOTE  crs: not exposed by xugrid, xugrid comparison skipped
PASS  depth: values and statistics identical
PASS  depth: fill/missing understood   masked in decoded new = 0, stored fill/NaN = 0, _FillValue = nan
PASS  depth: xugrid values and mean match original   xugrid mean = 484.262, original mean = 484.262
PASS  bottom_index_node: values and statistics identical
PASS  bottom_index_node: fill/missing understood   masked in decoded new = 0, stored fill/NaN = 0, _FillValue = None
PASS  bottom_index_node: xugrid values

SystemExit: 0

/lustre/env/miniconda3/envs/STOFS-Observer/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## 2. verticalVelocity files

**What will change** (besides the common changes and the shared 3D variables above)

| Variable | Changes |
| --- | --- |
| `verticalVelocity` | `standard_name = upward_sea_water_velocity`, `long_name = vertical velocity of sea water`, units set explicitly to `m s-1` (positive upward) |


In [2]:
import numpy as np
import xarray as xr

# ---- set your files here ---------------------------------------------------
src = "stofs_3d_atl.t12z.fields.verticalVelocity_f049_060.nc"
out = "stofs_3d_atl.t12z.fields.verticalVelocity_f049_060_compliant.nc"
title = "STOFS-3D-Atlantic verticalVelocity"
# ----------------------------------------------------------------------------

# The data variable is about 7 GB. With dask installed it is written one time step at a
# time (about 600 MB), so the whole variable is never held in memory. Without dask,
# writing needs one free block of about 7 GB and can fail with a MemoryError.
try:
    import dask
    dask.config.set(scheduler="synchronous")   # one chunk at a time keeps memory low
    chunks = {"time": 1}
except ImportError:
    chunks = None
ds = xr.open_dataset(src, mask_and_scale=False, decode_times=False, chunks=chunks)

# UGRID: mesh variable as a dimensionless int32 scalar, without the optional
# face/edge coordinate references (they conflict with CF's one lon/lat rule)
mesh_attrs = dict(ds["SCHISM_hgrid"].attrs)
mesh_attrs.pop("edge_coordinates", None)
mesh_attrs.pop("face_coordinates", None)
ds = ds.drop_vars("SCHISM_hgrid")
ds["SCHISM_hgrid"] = xr.Variable((), np.int32(0), mesh_attrs)

# UGRID: connectivity arrays (int32; fill value only on faces)
ds["SCHISM_hgrid_face_nodes"] = ds["SCHISM_hgrid_face_nodes"].astype("int32")
ds["SCHISM_hgrid_edge_nodes"] = ds["SCHISM_hgrid_edge_nodes"].astype("int32")
for v in ["SCHISM_hgrid_face_nodes", "SCHISM_hgrid_edge_nodes"]:
    ds[v].attrs.pop("_FillValue", None)
    ds[v].encoding.pop("_FillValue", None)

# CF: standard_name longitude/latitude on node coordinates only
ds["SCHISM_hgrid_node_x"].attrs.update(standard_name="longitude", long_name="node longitude", units="degrees_east")
ds["SCHISM_hgrid_node_y"].attrs.update(standard_name="latitude", long_name="node latitude", units="degrees_north")
for v in ["SCHISM_hgrid_face_x", "SCHISM_hgrid_face_y", "SCHISM_hgrid_edge_x", "SCHISM_hgrid_edge_y"]:
    if v in ds.variables:
        ds[v].attrs.pop("standard_name", None)
        ds[v].attrs.setdefault("long_name", v.replace("SCHISM_hgrid_", "").replace("_", " ") + " coordinate")

# CF: names and units (check these against your data)
ds["verticalVelocity"].attrs.update(standard_name="upward_sea_water_velocity", long_name="upward_sea_water_velocity", units="m", positive="up")
ds["depth"].attrs.update(standard_name="sea_floor_depth_below_geoid", long_name="bathymetry at nodes", units="m", positive="down")
ds["minimum_depth"].attrs.update(long_name="minimum water depth for wetting/drying", units="m")
ds["bottom_index_node"].attrs.update(long_name="bottom layer index at each node")
ds["verticalVelocity"].attrs["units"] = "m s-1"

# UGRID: tell readers which mesh and location each data variable lives on
for v in ["depth", "bottom_index_node", "verticalVelocity"]:
    ds[v].attrs["mesh"] = "SCHISM_hgrid"
    ds[v].attrs["location"] = "node"

# CF: fill values
ds["time"].attrs.pop("_FillValue", None)
ds["time"].encoding.pop("_FillValue", None)
for name in ds.variables:
    a = ds[name].attrs
    if "missing_value" in a:
        a.setdefault("_FillValue", a["missing_value"])
        del a["missing_value"]
ds["time"].attrs.pop("_FillValue", None)

# CF: time calendar stated explicitly (it is the default)
ds["time"].attrs["calendar"] = "standard"

# CF: global attributes
ds.attrs["Conventions"] = "CF-1.11 UGRID-1.0"
ds.attrs["title"] = title

ds.to_netcdf(out, encoding={
    "time": {"_FillValue": None},
    "SCHISM_hgrid_edge_nodes": {"_FillValue": None},
    "SCHISM_hgrid_face_nodes": {"dtype": "int32", "_FillValue": np.int32(-1)},
})
print("Wrote", out)

Wrote stofs_3d_atl.t12z.fields.verticalVelocity_f049_060_compliant.nc


**Test.** Compares the new file with the original: values, statistics, fill handling, connectivity, and a read through xugrid.


In [3]:
"""
Check that a *_compliant.nc file still holds the same data as the original:

 1. every numeric variable has identical values (NaN positions included)
 2. statistics (valid count, masked count, sum, mean, min, max) are identical
 3. fill / missing values are understood: the number of masked (NaN) values in
    the decoded new file equals the number of raw _FillValue (+ NaN) values
 4. the connectivity arrays are unchanged
 5. xugrid opens the new file, builds the mesh with the right sizes and
    connectivity, and a xugrid reduction (mean) matches xarray on the original
"""
import sys

import numpy as np
import xarray as xr
import xugrid as xu

# ---- set your files here ---------------------------------------------------
ORIG = "stofs_3d_atl.t12z.fields.verticalVelocity_f049_060.nc"
NEW = "stofs_3d_atl.t12z.fields.verticalVelocity_f049_060_compliant.nc"

# variables that were renamed in the new file (field2d), else leave empty:
# RENAMES = {"uvel4.5": "uvel_4p5m", "vvel4.5": "vvel_4p5m",
#            "uvel0.5": "uvel_0p5m", "vvel0.5": "vvel_0p5m"}
RENAMES = {}

# which time steps to test: None = all (slow on 8 GB files), or e.g. [0, -1]
TIME_STEPS = None
# ----------------------------------------------------------------------------

SKIP = {"SCHISM_hgrid"}   # placeholder variable; its type was changed on purpose
FACE_CANDIDATES = ["SCHISM_hgrid_face_nodes", "element"]
EDGE_CANDIDATES = ["SCHISM_hgrid_edge_nodes"]

results = []


def check(name, ok, msg=""):
    results.append((name, bool(ok)))
    print(("PASS  " if ok else "FAIL  ") + name + (f"   {msg}" if msg else ""))


class Stats:
    def __init__(self):
        self.n = 0          # valid values
        self.nnan = 0       # masked (NaN) values
        self.sum = 0.0
        self.min = np.inf
        self.max = -np.inf

    def add(self, a):
        if a.dtype.kind == "f":
            nan = int(np.isnan(a).sum())
            self.nnan += nan
            valid = a.size - nan
            if valid:
                self.n += valid
                self.sum += float(np.nansum(a, dtype=np.float64))
                self.min = min(self.min, float(np.nanmin(a)))
                self.max = max(self.max, float(np.nanmax(a)))
        else:
            self.n += a.size
            self.sum += float(a.sum(dtype=np.float64))
            self.min = min(self.min, float(a.min()))
            self.max = max(self.max, float(a.max()))

    @property
    def mean(self):
        return self.sum / self.n if self.n else float("nan")

    def same_as(self, o):
        return (self.n, self.nnan, self.sum, self.min, self.max) == \
               (o.n, o.nnan, o.sum, o.min, o.max)


def step_indices(da):
    """Time steps to test for a variable (None = the whole variable at once)."""
    if "time" in da.dims and da.ndim > 1:
        n = da.sizes["time"]
        return list(range(n)) if TIME_STEPS is None else sorted({i % n for i in TIME_STEPS})
    return [None]


def get(da, i):
    return da.values if i is None else da.isel(time=i).values


def as_xr(x):
    return getattr(x, "obj", x)   # unwrap xugrid objects


# ---- open everything --------------------------------------------------------
orig_dec = xr.open_dataset(ORIG, decode_times=False)                       # fills masked to NaN
new_dec = xr.open_dataset(NEW, decode_times=False)
orig_raw = xr.open_dataset(ORIG, mask_and_scale=False, decode_times=False)  # as stored
new_raw = xr.open_dataset(NEW, mask_and_scale=False, decode_times=False)
uds = xu.open_dataset(NEW)

face_name = next(n for n in FACE_CANDIDATES if n in orig_raw.variables)
edge_name = next((n for n in EDGE_CANDIDATES if n in orig_raw.variables), None)
conn_names = {face_name} | ({edge_name} if edge_name else set())

# ---- 1-3. every numeric variable --------------------------------------------
print("\n=== Variables: values, statistics, fill/missing handling ===")
rows = []
for v in orig_dec.variables:
    if v in SKIP or v in conn_names:
        continue
    od = orig_dec[v]
    if od.dtype.kind not in "fiu":
        continue
    rn = RENAMES.get(v, v)
    if rn not in new_dec.variables:
        check(v, False, "missing in new file")
        continue
    nd, nr = new_dec[rn], new_raw[rn]
    if od.dims != nd.dims or od.shape != nd.shape:
        check(v, False, f"dims/shape differ: {od.dims}{od.shape} vs {nd.dims}{nd.shape}")
        continue

    so, sn = Stats(), Stats()
    identical = True
    n_fill_raw = n_nan_raw = 0
    fill = nr.attrs.get("_FillValue")
    for i in step_indices(od):
        a_o, a_n, raw = get(od, i), get(nd, i), get(nr, i)
        identical &= np.array_equal(a_o, a_n, equal_nan=(a_o.dtype.kind == "f"))
        so.add(a_o)
        sn.add(a_n)
        if fill is not None:
            n_fill_raw += int((raw == fill).sum())
        if raw.dtype.kind == "f":
            n_nan_raw += int(np.isnan(raw).sum())

    ok_vals = identical and so.same_as(sn)
    expected_masked = n_fill_raw + n_nan_raw
    ok_fill = sn.nnan == expected_masked
    ok_attr = ("missing_value" not in nr.attrs) or (nr.attrs.get("missing_value") == fill)
    check(f"{v}: values and statistics identical", ok_vals,
          "" if ok_vals else f"orig n={so.n} sum={so.sum} | new n={sn.n} sum={sn.sum}")
    check(f"{v}: fill/missing understood", ok_fill and ok_attr,
          f"masked in decoded new = {sn.nnan}, stored fill/NaN = {expected_masked}, "
          f"_FillValue = {fill}")

    # xugrid reading of the same variable (first tested time step)
    if rn != "time" and rn not in as_xr(uds).variables:
        # xugrid does not expose every variable (e.g. the "crs" grid-mapping
        # variable); the xarray checks above already cover it
        print(f"NOTE  {v}: not exposed by xugrid, xugrid comparison skipped")
    elif rn != "time":
        try:
            i0 = step_indices(od)[0]
            ua = uds[rn]
            xa = as_xr(ua)
            a_u = get(xa, i0)
            res_u = ua if i0 is None else ua.isel(time=i0)
            mean_u = float(as_xr(res_u.mean()))
            od0 = od if i0 is None else od.isel(time=i0)
            mean_o = float(od0.mean())
            ok_u = np.array_equal(a_u, get(od, i0), equal_nan=(a_u.dtype.kind == "f")) \
                and np.isclose(mean_u, mean_o, rtol=1e-6, atol=0, equal_nan=True)
            check(f"{v}: xugrid values and mean match original", ok_u,
                  f"xugrid mean = {mean_u:.6g}, original mean = {mean_o:.6g}")
        except Exception as e:  # report, don't stop
            check(f"{v}: xugrid values and mean match original", False, repr(e))

    rows.append((v, str(od.dtype), sn.n, sn.nnan, sn.mean, sn.min, sn.max))

# ---- 4. connectivity arrays (as stored) -------------------------------------
print("\n=== Connectivity arrays ===")
for name in sorted(conn_names):
    fo = np.nan_to_num(orig_raw[name].values.astype("float64"), nan=-1).astype("int64")
    fn = new_raw[name].values.astype("int64")
    check(f"{name}: unchanged as stored", np.array_equal(fo, fn))

# ---- 5. xugrid mesh ---------------------------------------------------------
print("\n=== xugrid mesh ===")
grid = uds.ugrid.grids[0]
mesh_var = next(n for n, x in new_raw.variables.items()
                if x.attrs.get("cf_role") == "mesh_topology")
node_x_name = new_raw[mesh_var].attrs["node_coordinates"].split()[0]
fn = new_raw[face_name].values.astype("int64")
start = int(new_raw[face_name].attrs.get("start_index", 0))
check("mesh variable present", True, f"'{mesh_var}'")
check("n_node matches", grid.n_node == new_raw[node_x_name].size,
      f"xugrid {grid.n_node}, file {new_raw[node_x_name].size}")
check("n_face matches", grid.n_face == fn.shape[0],
      f"xugrid {grid.n_face}, file {fn.shape[0]}")
expected = np.where(fn >= start, fn - start, -1)   # xugrid uses 0-based indices
gc = np.asarray(grid.face_node_connectivity)
check("face connectivity read correctly by xugrid",
      gc.shape == expected.shape and np.array_equal(gc, expected),
      f"shape {gc.shape} vs expected {expected.shape}")

# ---- summary ----------------------------------------------------------------
print("\n=== Statistics of the new file (decoded; fill values excluded) ===")
print(f"{'variable':28s}{'dtype':9s}{'valid':>14s}{'masked':>12s}{'mean':>14s}{'min':>14s}{'max':>14s}")
for r in rows:
    print(f"{r[0]:28s}{r[1]:9s}{r[2]:14d}{r[3]:12d}{r[4]:14.6g}{r[5]:14.6g}{r[6]:14.6g}")

failed = [n for n, ok in results if not ok]
print(f"\n{len(results) - len(failed)} passed, {len(failed)} failed")
for n in failed:
    print("  FAILED:", n)
sys.exit(1 if failed else 0)


=== Variables: values, statistics, fill/missing handling ===
PASS  minimum_depth: values and statistics identical
PASS  minimum_depth: fill/missing understood   masked in decoded new = 0, stored fill/NaN = 0, _FillValue = nan
PASS  minimum_depth: xugrid values and mean match original   xugrid mean = 0.0001, original mean = 0.0001
PASS  crs: values and statistics identical
PASS  crs: fill/missing understood   masked in decoded new = 0, stored fill/NaN = 0, _FillValue = None
NOTE  crs: not exposed by xugrid, xugrid comparison skipped
PASS  depth: values and statistics identical
PASS  depth: fill/missing understood   masked in decoded new = 0, stored fill/NaN = 0, _FillValue = nan
PASS  depth: xugrid values and mean match original   xugrid mean = 484.262, original mean = 484.262
PASS  bottom_index_node: values and statistics identical
PASS  bottom_index_node: fill/missing understood   masked in decoded new = 0, stored fill/NaN = 0, _FillValue = None
PASS  bottom_index_node: xugrid values

SystemExit: 0

/lustre/env/miniconda3/envs/STOFS-Observer/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## 3. temperature files

**What will change** (besides the common changes and the shared 3D variables above)

| Variable | Changes |
| --- | --- |
| `temperature` | `standard_name = sea_water_temperature`, `long_name = sea water temperature`, units set to `degree_Celsius` (an earlier NCO step wrote a free-text unit), `units_metadata = "temperature: on_scale"` |


In [4]:
import numpy as np
import xarray as xr

# ---- set your files here ---------------------------------------------------
src = "stofs_3d_atl.t12z.fields.temperature_f025_036.nc"
out = "stofs_3d_atl.t12z.fields.temperature_f025_036_compliant.nc"
# title comes from the file name, e.g. "STOFS-3D-Atlantic temperature_f073_084"
title = "STOFS-3D-Atlantic temperature"
# ----------------------------------------------------------------------------

# The data variable is about 7 GB. With dask installed it is written one time step at a
# time (about 600 MB), so the whole variable is never held in memory. Without dask,
# writing needs one free block of about 7 GB and can fail with a MemoryError.
try:
    import dask
    dask.config.set(scheduler="synchronous")   # one chunk at a time keeps memory low
    chunks = {"time": 1}
except ImportError:
    chunks = None
ds = xr.open_dataset(src, mask_and_scale=False, decode_times=False, chunks=chunks)

# ===== UGRID (same as the other scripts) =====================================
# mesh variable as a dimensionless int32 scalar, without the optional
# face/edge coordinate references (they conflict with CF's one lon/lat rule)
mesh_attrs = dict(ds["SCHISM_hgrid"].attrs)
mesh_attrs.pop("edge_coordinates", None)
mesh_attrs.pop("face_coordinates", None)
ds = ds.drop_vars("SCHISM_hgrid")
ds["SCHISM_hgrid"] = xr.Variable((), np.int32(0), mesh_attrs)

# connectivity arrays (int32; fill value only on faces)
ds["SCHISM_hgrid_face_nodes"] = ds["SCHISM_hgrid_face_nodes"].astype("int32")
ds["SCHISM_hgrid_edge_nodes"] = ds["SCHISM_hgrid_edge_nodes"].astype("int32")
for v in ["SCHISM_hgrid_face_nodes", "SCHISM_hgrid_edge_nodes"]:
    ds[v].attrs.pop("_FillValue", None)
    ds[v].encoding.pop("_FillValue", None)

# ===== CF: coordinates ========================================================
ds["SCHISM_hgrid_node_x"].attrs.update(standard_name="longitude", long_name="node longitude", units="degrees_east")
ds["SCHISM_hgrid_node_y"].attrs.update(standard_name="latitude", long_name="node latitude", units="degrees_north")
for v in ["SCHISM_hgrid_face_x", "SCHISM_hgrid_face_y", "SCHISM_hgrid_edge_x", "SCHISM_hgrid_edge_y"]:
    if v in ds.variables:
        ds[v].attrs.pop("standard_name", None)
        ds[v].attrs.setdefault("long_name", v.replace("SCHISM_hgrid_", "").replace("_", " ") + " coordinate")

# ===== CF: names and units for THIS file (check against your data) ===========
ds["depth"].attrs.update(standard_name="sea_floor_depth_below_geoid", long_name="bathymetry at nodes", units="m", positive="down")
ds["minimum_depth"].attrs.update(long_name="minimum water depth for wetting/drying", units="m")
ds["bottom_index_node"].attrs.update(long_name="bottom layer index at each node")
ds["temperature"].attrs.update(
    standard_name="sea_water_temperature",
    long_name="sea water temperature",
    units="degree_Celsius",   # overwrites the units NCO set; see notes
)
ds["temperature"].attrs["units_metadata"] = "temperature: on_scale"

# UGRID: tell readers which mesh and location each data variable lives on
for v in ["depth", "bottom_index_node", "temperature"]:
    ds[v].attrs["mesh"] = "SCHISM_hgrid"
    ds[v].attrs["location"] = "node"

# ===== CF: fill values =======================================================
# missing_value must equal _FillValue: keep a single attribute
for name in ds.variables:
    a = ds[name].attrs
    if "missing_value" in a:
        a.setdefault("_FillValue", a["missing_value"])
        del a["missing_value"]
# coordinate variables must not have _FillValue
ds["time"].attrs.pop("_FillValue", None)
ds["time"].encoding.pop("_FillValue", None)

# CF: time calendar stated explicitly (it is the default)
ds["time"].attrs["calendar"] = "standard"

# ===== CF: global attributes =================================================
ds.attrs["Conventions"] = "CF-1.11 UGRID-1.0"
ds.attrs["title"] = title

ds.to_netcdf(out, encoding={
    "time": {"_FillValue": None},
    "SCHISM_hgrid_edge_nodes": {"_FillValue": None},
    "SCHISM_hgrid_face_nodes": {"dtype": "int32", "_FillValue": np.int32(-1)},
})
print("Wrote", out)

Wrote stofs_3d_atl.t12z.fields.temperature_f025_036_compliant.nc


**Test.** Compares the new file with the original: values, statistics, fill handling, connectivity, and a read through xugrid.


In [5]:
"""
Check that a *_compliant.nc file still holds the same data as the original:

 1. every numeric variable has identical values (NaN positions included)
 2. statistics (valid count, masked count, sum, mean, min, max) are identical
 3. fill / missing values are understood: the number of masked (NaN) values in
    the decoded new file equals the number of raw _FillValue (+ NaN) values
 4. the connectivity arrays are unchanged
 5. xugrid opens the new file, builds the mesh with the right sizes and
    connectivity, and a xugrid reduction (mean) matches xarray on the original
"""
import sys

import numpy as np
import xarray as xr
import xugrid as xu

# ---- set your files here ---------------------------------------------------
ORIG = "stofs_3d_atl.t12z.fields.temperature_f025_036.nc"
NEW = "stofs_3d_atl.t12z.fields.temperature_f025_036_compliant.nc"

# variables that were renamed in the new file (field2d), else leave empty:
# RENAMES = {"uvel4.5": "uvel_4p5m", "vvel4.5": "vvel_4p5m",
#            "uvel0.5": "uvel_0p5m", "vvel0.5": "vvel_0p5m"}
RENAMES = {}

# which time steps to test: None = all (slow on 8 GB files), or e.g. [0, -1]
TIME_STEPS = None
# ----------------------------------------------------------------------------

SKIP = {"SCHISM_hgrid"}   # placeholder variable; its type was changed on purpose
FACE_CANDIDATES = ["SCHISM_hgrid_face_nodes", "element"]
EDGE_CANDIDATES = ["SCHISM_hgrid_edge_nodes"]

results = []


def check(name, ok, msg=""):
    results.append((name, bool(ok)))
    print(("PASS  " if ok else "FAIL  ") + name + (f"   {msg}" if msg else ""))


class Stats:
    def __init__(self):
        self.n = 0          # valid values
        self.nnan = 0       # masked (NaN) values
        self.sum = 0.0
        self.min = np.inf
        self.max = -np.inf

    def add(self, a):
        if a.dtype.kind == "f":
            nan = int(np.isnan(a).sum())
            self.nnan += nan
            valid = a.size - nan
            if valid:
                self.n += valid
                self.sum += float(np.nansum(a, dtype=np.float64))
                self.min = min(self.min, float(np.nanmin(a)))
                self.max = max(self.max, float(np.nanmax(a)))
        else:
            self.n += a.size
            self.sum += float(a.sum(dtype=np.float64))
            self.min = min(self.min, float(a.min()))
            self.max = max(self.max, float(a.max()))

    @property
    def mean(self):
        return self.sum / self.n if self.n else float("nan")

    def same_as(self, o):
        return (self.n, self.nnan, self.sum, self.min, self.max) == \
               (o.n, o.nnan, o.sum, o.min, o.max)


def step_indices(da):
    """Time steps to test for a variable (None = the whole variable at once)."""
    if "time" in da.dims and da.ndim > 1:
        n = da.sizes["time"]
        return list(range(n)) if TIME_STEPS is None else sorted({i % n for i in TIME_STEPS})
    return [None]


def get(da, i):
    return da.values if i is None else da.isel(time=i).values


def as_xr(x):
    return getattr(x, "obj", x)   # unwrap xugrid objects


# ---- open everything --------------------------------------------------------
orig_dec = xr.open_dataset(ORIG, decode_times=False)                       # fills masked to NaN
new_dec = xr.open_dataset(NEW, decode_times=False)
orig_raw = xr.open_dataset(ORIG, mask_and_scale=False, decode_times=False)  # as stored
new_raw = xr.open_dataset(NEW, mask_and_scale=False, decode_times=False)
uds = xu.open_dataset(NEW)

face_name = next(n for n in FACE_CANDIDATES if n in orig_raw.variables)
edge_name = next((n for n in EDGE_CANDIDATES if n in orig_raw.variables), None)
conn_names = {face_name} | ({edge_name} if edge_name else set())

# ---- 1-3. every numeric variable --------------------------------------------
print("\n=== Variables: values, statistics, fill/missing handling ===")
rows = []
for v in orig_dec.variables:
    if v in SKIP or v in conn_names:
        continue
    od = orig_dec[v]
    if od.dtype.kind not in "fiu":
        continue
    rn = RENAMES.get(v, v)
    if rn not in new_dec.variables:
        check(v, False, "missing in new file")
        continue
    nd, nr = new_dec[rn], new_raw[rn]
    if od.dims != nd.dims or od.shape != nd.shape:
        check(v, False, f"dims/shape differ: {od.dims}{od.shape} vs {nd.dims}{nd.shape}")
        continue

    so, sn = Stats(), Stats()
    identical = True
    n_fill_raw = n_nan_raw = 0
    fill = nr.attrs.get("_FillValue")
    for i in step_indices(od):
        a_o, a_n, raw = get(od, i), get(nd, i), get(nr, i)
        identical &= np.array_equal(a_o, a_n, equal_nan=(a_o.dtype.kind == "f"))
        so.add(a_o)
        sn.add(a_n)
        if fill is not None:
            n_fill_raw += int((raw == fill).sum())
        if raw.dtype.kind == "f":
            n_nan_raw += int(np.isnan(raw).sum())

    ok_vals = identical and so.same_as(sn)
    expected_masked = n_fill_raw + n_nan_raw
    ok_fill = sn.nnan == expected_masked
    ok_attr = ("missing_value" not in nr.attrs) or (nr.attrs.get("missing_value") == fill)
    check(f"{v}: values and statistics identical", ok_vals,
          "" if ok_vals else f"orig n={so.n} sum={so.sum} | new n={sn.n} sum={sn.sum}")
    check(f"{v}: fill/missing understood", ok_fill and ok_attr,
          f"masked in decoded new = {sn.nnan}, stored fill/NaN = {expected_masked}, "
          f"_FillValue = {fill}")

    # xugrid reading of the same variable (first tested time step)
    if rn != "time" and rn not in as_xr(uds).variables:
        # xugrid does not expose every variable (e.g. the "crs" grid-mapping
        # variable); the xarray checks above already cover it
        print(f"NOTE  {v}: not exposed by xugrid, xugrid comparison skipped")
    elif rn != "time":
        try:
            i0 = step_indices(od)[0]
            ua = uds[rn]
            xa = as_xr(ua)
            a_u = get(xa, i0)
            res_u = ua if i0 is None else ua.isel(time=i0)
            mean_u = float(as_xr(res_u.mean()))
            od0 = od if i0 is None else od.isel(time=i0)
            mean_o = float(od0.mean())
            ok_u = np.array_equal(a_u, get(od, i0), equal_nan=(a_u.dtype.kind == "f")) \
                and np.isclose(mean_u, mean_o, rtol=1e-6, atol=0, equal_nan=True)
            check(f"{v}: xugrid values and mean match original", ok_u,
                  f"xugrid mean = {mean_u:.6g}, original mean = {mean_o:.6g}")
        except Exception as e:  # report, don't stop
            check(f"{v}: xugrid values and mean match original", False, repr(e))

    rows.append((v, str(od.dtype), sn.n, sn.nnan, sn.mean, sn.min, sn.max))

# ---- 4. connectivity arrays (as stored) -------------------------------------
print("\n=== Connectivity arrays ===")
for name in sorted(conn_names):
    fo = np.nan_to_num(orig_raw[name].values.astype("float64"), nan=-1).astype("int64")
    fn = new_raw[name].values.astype("int64")
    check(f"{name}: unchanged as stored", np.array_equal(fo, fn))

# ---- 5. xugrid mesh ---------------------------------------------------------
print("\n=== xugrid mesh ===")
grid = uds.ugrid.grids[0]
mesh_var = next(n for n, x in new_raw.variables.items()
                if x.attrs.get("cf_role") == "mesh_topology")
node_x_name = new_raw[mesh_var].attrs["node_coordinates"].split()[0]
fn = new_raw[face_name].values.astype("int64")
start = int(new_raw[face_name].attrs.get("start_index", 0))
check("mesh variable present", True, f"'{mesh_var}'")
check("n_node matches", grid.n_node == new_raw[node_x_name].size,
      f"xugrid {grid.n_node}, file {new_raw[node_x_name].size}")
check("n_face matches", grid.n_face == fn.shape[0],
      f"xugrid {grid.n_face}, file {fn.shape[0]}")
expected = np.where(fn >= start, fn - start, -1)   # xugrid uses 0-based indices
gc = np.asarray(grid.face_node_connectivity)
check("face connectivity read correctly by xugrid",
      gc.shape == expected.shape and np.array_equal(gc, expected),
      f"shape {gc.shape} vs expected {expected.shape}")

# ---- summary ----------------------------------------------------------------
print("\n=== Statistics of the new file (decoded; fill values excluded) ===")
print(f"{'variable':28s}{'dtype':9s}{'valid':>14s}{'masked':>12s}{'mean':>14s}{'min':>14s}{'max':>14s}")
for r in rows:
    print(f"{r[0]:28s}{r[1]:9s}{r[2]:14d}{r[3]:12d}{r[4]:14.6g}{r[5]:14.6g}{r[6]:14.6g}")

failed = [n for n, ok in results if not ok]
print(f"\n{len(results) - len(failed)} passed, {len(failed)} failed")
for n in failed:
    print("  FAILED:", n)
sys.exit(1 if failed else 0)


=== Variables: values, statistics, fill/missing handling ===
PASS  minimum_depth: values and statistics identical
PASS  minimum_depth: fill/missing understood   masked in decoded new = 0, stored fill/NaN = 0, _FillValue = nan
PASS  minimum_depth: xugrid values and mean match original   xugrid mean = 0.0001, original mean = 0.0001
PASS  crs: values and statistics identical
PASS  crs: fill/missing understood   masked in decoded new = 0, stored fill/NaN = 0, _FillValue = None
NOTE  crs: not exposed by xugrid, xugrid comparison skipped
PASS  depth: values and statistics identical
PASS  depth: fill/missing understood   masked in decoded new = 0, stored fill/NaN = 0, _FillValue = nan
PASS  depth: xugrid values and mean match original   xugrid mean = 484.262, original mean = 484.262
PASS  bottom_index_node: values and statistics identical
PASS  bottom_index_node: fill/missing understood   masked in decoded new = 0, stored fill/NaN = 0, _FillValue = None
PASS  bottom_index_node: xugrid values

SystemExit: 0

/lustre/env/miniconda3/envs/STOFS-Observer/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## 4. salinity files

**What will change** (besides the common changes and the shared 3D variables above)

| Variable | Changes |
| --- | --- |
| `salinity` | `standard_name = sea_water_practical_salinity`, `long_name = sea water practical salinity (PSU)`, units `PSU` changed to `1` because PSU is not a UDUNITS unit |


In [6]:
import numpy as np
import xarray as xr

# ---- set your files here ---------------------------------------------------
src = "stofs_3d_atl.t12z.fields.salinity_f037_048.nc"
out = "stofs_3d_atl.t12z.fields.salinity_f037_048_compliant.nc"
# title comes from the file name, e.g. "STOFS-3D-Atlantic salinity_n001_012"
title = "STOFS-3D-Atlantic salinity"
# ----------------------------------------------------------------------------

# The data variable is about 7 GB. With dask installed it is written one time step at a
# time (about 600 MB), so the whole variable is never held in memory. Without dask,
# writing needs one free block of about 7 GB and can fail with a MemoryError.
try:
    import dask
    dask.config.set(scheduler="synchronous")   # one chunk at a time keeps memory low
    chunks = {"time": 1}
except ImportError:
    chunks = None
ds = xr.open_dataset(src, mask_and_scale=False, decode_times=False, chunks=chunks)

# ===== UGRID (same as the other scripts) =====================================
# mesh variable as a dimensionless int32 scalar, without the optional
# face/edge coordinate references (they conflict with CF's one lon/lat rule)
mesh_attrs = dict(ds["SCHISM_hgrid"].attrs)
mesh_attrs.pop("edge_coordinates", None)
mesh_attrs.pop("face_coordinates", None)
ds = ds.drop_vars("SCHISM_hgrid")
ds["SCHISM_hgrid"] = xr.Variable((), np.int32(0), mesh_attrs)

# connectivity arrays (int32; fill value only on faces)
ds["SCHISM_hgrid_face_nodes"] = ds["SCHISM_hgrid_face_nodes"].astype("int32")
ds["SCHISM_hgrid_edge_nodes"] = ds["SCHISM_hgrid_edge_nodes"].astype("int32")
for v in ["SCHISM_hgrid_face_nodes", "SCHISM_hgrid_edge_nodes"]:
    ds[v].attrs.pop("_FillValue", None)
    ds[v].encoding.pop("_FillValue", None)

# ===== CF: coordinates ========================================================
ds["SCHISM_hgrid_node_x"].attrs.update(standard_name="longitude", long_name="node longitude", units="degrees_east")
ds["SCHISM_hgrid_node_y"].attrs.update(standard_name="latitude", long_name="node latitude", units="degrees_north")
for v in ["SCHISM_hgrid_face_x", "SCHISM_hgrid_face_y", "SCHISM_hgrid_edge_x", "SCHISM_hgrid_edge_y"]:
    if v in ds.variables:
        ds[v].attrs.pop("standard_name", None)
        ds[v].attrs.setdefault("long_name", v.replace("SCHISM_hgrid_", "").replace("_", " ") + " coordinate")

# ===== CF: names and units for THIS file (check against your data) ===========
ds["depth"].attrs.update(standard_name="sea_floor_depth_below_geoid", long_name="bathymetry at nodes", units="m", positive="down")
ds["minimum_depth"].attrs.update(long_name="minimum water depth for wetting/drying", units="m")
ds["bottom_index_node"].attrs.update(long_name="bottom layer index at each node")
# "PSU" is not a UDUNITS unit. Practical salinity is dimensionless in CF, so
# its units are "1". The PSU labelling is kept in long_name.
ds["salinity"].attrs.update(
    standard_name="sea_water_practical_salinity",
    long_name="sea water practical salinity (PSU)",
    units="1",
)

# UGRID: tell readers which mesh and location each data variable lives on
for v in ["depth", "bottom_index_node", "salinity"]:
    ds[v].attrs["mesh"] = "SCHISM_hgrid"
    ds[v].attrs["location"] = "node"

# ===== CF: fill values =======================================================
# missing_value must equal _FillValue: keep a single attribute
for name in ds.variables:
    a = ds[name].attrs
    if "missing_value" in a:
        a.setdefault("_FillValue", a["missing_value"])
        del a["missing_value"]
# coordinate variables must not have _FillValue
ds["time"].attrs.pop("_FillValue", None)
ds["time"].encoding.pop("_FillValue", None)

# CF: time calendar stated explicitly (it is the default)
ds["time"].attrs["calendar"] = "standard"

# ===== CF: global attributes =================================================
ds.attrs["Conventions"] = "CF-1.11 UGRID-1.0"
ds.attrs["title"] = title

ds.to_netcdf(out, encoding={
    "time": {"_FillValue": None},
    "SCHISM_hgrid_edge_nodes": {"_FillValue": None},
    "SCHISM_hgrid_face_nodes": {"dtype": "int32", "_FillValue": np.int32(-1)},
})
print("Wrote", out)

Wrote stofs_3d_atl.t12z.fields.salinity_f037_048_compliant.nc


**Test.** Compares the new file with the original: values, statistics, fill handling, connectivity, and a read through xugrid.


In [7]:
"""
Check that a *_compliant.nc file still holds the same data as the original:

 1. every numeric variable has identical values (NaN positions included)
 2. statistics (valid count, masked count, sum, mean, min, max) are identical
 3. fill / missing values are understood: the number of masked (NaN) values in
    the decoded new file equals the number of raw _FillValue (+ NaN) values
 4. the connectivity arrays are unchanged
 5. xugrid opens the new file, builds the mesh with the right sizes and
    connectivity, and a xugrid reduction (mean) matches xarray on the original
"""
import sys

import numpy as np
import xarray as xr
import xugrid as xu

# ---- set your files here ---------------------------------------------------
ORIG = "stofs_3d_atl.t12z.fields.salinity_f037_048.nc"
NEW = "stofs_3d_atl.t12z.fields.salinity_f037_048_compliant.nc"

# variables that were renamed in the new file (field2d), else leave empty:
# RENAMES = {"uvel4.5": "uvel_4p5m", "vvel4.5": "vvel_4p5m",
#            "uvel0.5": "uvel_0p5m", "vvel0.5": "vvel_0p5m"}
RENAMES = {}

# which time steps to test: None = all (slow on 8 GB files), or e.g. [0, -1]
TIME_STEPS = None
# ----------------------------------------------------------------------------

SKIP = {"SCHISM_hgrid"}   # placeholder variable; its type was changed on purpose
FACE_CANDIDATES = ["SCHISM_hgrid_face_nodes", "element"]
EDGE_CANDIDATES = ["SCHISM_hgrid_edge_nodes"]

results = []


def check(name, ok, msg=""):
    results.append((name, bool(ok)))
    print(("PASS  " if ok else "FAIL  ") + name + (f"   {msg}" if msg else ""))


class Stats:
    def __init__(self):
        self.n = 0          # valid values
        self.nnan = 0       # masked (NaN) values
        self.sum = 0.0
        self.min = np.inf
        self.max = -np.inf

    def add(self, a):
        if a.dtype.kind == "f":
            nan = int(np.isnan(a).sum())
            self.nnan += nan
            valid = a.size - nan
            if valid:
                self.n += valid
                self.sum += float(np.nansum(a, dtype=np.float64))
                self.min = min(self.min, float(np.nanmin(a)))
                self.max = max(self.max, float(np.nanmax(a)))
        else:
            self.n += a.size
            self.sum += float(a.sum(dtype=np.float64))
            self.min = min(self.min, float(a.min()))
            self.max = max(self.max, float(a.max()))

    @property
    def mean(self):
        return self.sum / self.n if self.n else float("nan")

    def same_as(self, o):
        return (self.n, self.nnan, self.sum, self.min, self.max) == \
               (o.n, o.nnan, o.sum, o.min, o.max)


def step_indices(da):
    """Time steps to test for a variable (None = the whole variable at once)."""
    if "time" in da.dims and da.ndim > 1:
        n = da.sizes["time"]
        return list(range(n)) if TIME_STEPS is None else sorted({i % n for i in TIME_STEPS})
    return [None]


def get(da, i):
    return da.values if i is None else da.isel(time=i).values


def as_xr(x):
    return getattr(x, "obj", x)   # unwrap xugrid objects


# ---- open everything --------------------------------------------------------
orig_dec = xr.open_dataset(ORIG, decode_times=False)                       # fills masked to NaN
new_dec = xr.open_dataset(NEW, decode_times=False)
orig_raw = xr.open_dataset(ORIG, mask_and_scale=False, decode_times=False)  # as stored
new_raw = xr.open_dataset(NEW, mask_and_scale=False, decode_times=False)
uds = xu.open_dataset(NEW)

face_name = next(n for n in FACE_CANDIDATES if n in orig_raw.variables)
edge_name = next((n for n in EDGE_CANDIDATES if n in orig_raw.variables), None)
conn_names = {face_name} | ({edge_name} if edge_name else set())

# ---- 1-3. every numeric variable --------------------------------------------
print("\n=== Variables: values, statistics, fill/missing handling ===")
rows = []
for v in orig_dec.variables:
    if v in SKIP or v in conn_names:
        continue
    od = orig_dec[v]
    if od.dtype.kind not in "fiu":
        continue
    rn = RENAMES.get(v, v)
    if rn not in new_dec.variables:
        check(v, False, "missing in new file")
        continue
    nd, nr = new_dec[rn], new_raw[rn]
    if od.dims != nd.dims or od.shape != nd.shape:
        check(v, False, f"dims/shape differ: {od.dims}{od.shape} vs {nd.dims}{nd.shape}")
        continue

    so, sn = Stats(), Stats()
    identical = True
    n_fill_raw = n_nan_raw = 0
    fill = nr.attrs.get("_FillValue")
    for i in step_indices(od):
        a_o, a_n, raw = get(od, i), get(nd, i), get(nr, i)
        identical &= np.array_equal(a_o, a_n, equal_nan=(a_o.dtype.kind == "f"))
        so.add(a_o)
        sn.add(a_n)
        if fill is not None:
            n_fill_raw += int((raw == fill).sum())
        if raw.dtype.kind == "f":
            n_nan_raw += int(np.isnan(raw).sum())

    ok_vals = identical and so.same_as(sn)
    expected_masked = n_fill_raw + n_nan_raw
    ok_fill = sn.nnan == expected_masked
    ok_attr = ("missing_value" not in nr.attrs) or (nr.attrs.get("missing_value") == fill)
    check(f"{v}: values and statistics identical", ok_vals,
          "" if ok_vals else f"orig n={so.n} sum={so.sum} | new n={sn.n} sum={sn.sum}")
    check(f"{v}: fill/missing understood", ok_fill and ok_attr,
          f"masked in decoded new = {sn.nnan}, stored fill/NaN = {expected_masked}, "
          f"_FillValue = {fill}")

    # xugrid reading of the same variable (first tested time step)
    if rn != "time" and rn not in as_xr(uds).variables:
        # xugrid does not expose every variable (e.g. the "crs" grid-mapping
        # variable); the xarray checks above already cover it
        print(f"NOTE  {v}: not exposed by xugrid, xugrid comparison skipped")
    elif rn != "time":
        try:
            i0 = step_indices(od)[0]
            ua = uds[rn]
            xa = as_xr(ua)
            a_u = get(xa, i0)
            res_u = ua if i0 is None else ua.isel(time=i0)
            mean_u = float(as_xr(res_u.mean()))
            od0 = od if i0 is None else od.isel(time=i0)
            mean_o = float(od0.mean())
            ok_u = np.array_equal(a_u, get(od, i0), equal_nan=(a_u.dtype.kind == "f")) \
                and np.isclose(mean_u, mean_o, rtol=1e-6, atol=0, equal_nan=True)
            check(f"{v}: xugrid values and mean match original", ok_u,
                  f"xugrid mean = {mean_u:.6g}, original mean = {mean_o:.6g}")
        except Exception as e:  # report, don't stop
            check(f"{v}: xugrid values and mean match original", False, repr(e))

    rows.append((v, str(od.dtype), sn.n, sn.nnan, sn.mean, sn.min, sn.max))

# ---- 4. connectivity arrays (as stored) -------------------------------------
print("\n=== Connectivity arrays ===")
for name in sorted(conn_names):
    fo = np.nan_to_num(orig_raw[name].values.astype("float64"), nan=-1).astype("int64")
    fn = new_raw[name].values.astype("int64")
    check(f"{name}: unchanged as stored", np.array_equal(fo, fn))

# ---- 5. xugrid mesh ---------------------------------------------------------
print("\n=== xugrid mesh ===")
grid = uds.ugrid.grids[0]
mesh_var = next(n for n, x in new_raw.variables.items()
                if x.attrs.get("cf_role") == "mesh_topology")
node_x_name = new_raw[mesh_var].attrs["node_coordinates"].split()[0]
fn = new_raw[face_name].values.astype("int64")
start = int(new_raw[face_name].attrs.get("start_index", 0))
check("mesh variable present", True, f"'{mesh_var}'")
check("n_node matches", grid.n_node == new_raw[node_x_name].size,
      f"xugrid {grid.n_node}, file {new_raw[node_x_name].size}")
check("n_face matches", grid.n_face == fn.shape[0],
      f"xugrid {grid.n_face}, file {fn.shape[0]}")
expected = np.where(fn >= start, fn - start, -1)   # xugrid uses 0-based indices
gc = np.asarray(grid.face_node_connectivity)
check("face connectivity read correctly by xugrid",
      gc.shape == expected.shape and np.array_equal(gc, expected),
      f"shape {gc.shape} vs expected {expected.shape}")

# ---- summary ----------------------------------------------------------------
print("\n=== Statistics of the new file (decoded; fill values excluded) ===")
print(f"{'variable':28s}{'dtype':9s}{'valid':>14s}{'masked':>12s}{'mean':>14s}{'min':>14s}{'max':>14s}")
for r in rows:
    print(f"{r[0]:28s}{r[1]:9s}{r[2]:14d}{r[3]:12d}{r[4]:14.6g}{r[5]:14.6g}{r[6]:14.6g}")

failed = [n for n, ok in results if not ok]
print(f"\n{len(results) - len(failed)} passed, {len(failed)} failed")
for n in failed:
    print("  FAILED:", n)
sys.exit(1 if failed else 0)


=== Variables: values, statistics, fill/missing handling ===
PASS  minimum_depth: values and statistics identical
PASS  minimum_depth: fill/missing understood   masked in decoded new = 0, stored fill/NaN = 0, _FillValue = nan
PASS  minimum_depth: xugrid values and mean match original   xugrid mean = 0.0001, original mean = 0.0001
PASS  crs: values and statistics identical
PASS  crs: fill/missing understood   masked in decoded new = 0, stored fill/NaN = 0, _FillValue = None
NOTE  crs: not exposed by xugrid, xugrid comparison skipped
PASS  depth: values and statistics identical
PASS  depth: fill/missing understood   masked in decoded new = 0, stored fill/NaN = 0, _FillValue = nan
PASS  depth: xugrid values and mean match original   xugrid mean = 484.262, original mean = 484.262
PASS  bottom_index_node: values and statistics identical
PASS  bottom_index_node: fill/missing understood   masked in decoded new = 0, stored fill/NaN = 0, _FillValue = None
PASS  bottom_index_node: xugrid values

SystemExit: 0

/lustre/env/miniconda3/envs/STOFS-Observer/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## 5. horizontalVelX files

**What will change** (besides the common changes and the shared 3D variables above)

| Variable | Changes |
| --- | --- |
| `horizontalVelX` | `standard_name = eastward_sea_water_velocity`, `long_name = eastward sea water velocity`, units set explicitly to `m s-1` (assumes the east component) |


In [8]:
import numpy as np
import xarray as xr

# ---- set your files here ---------------------------------------------------
# Works for both horizontalVelX and horizontalVelY files.
src = "stofs_3d_atl.t12z.fields.horizontalVelX_f049_060.nc"
out = "stofs_3d_atl.t12z.fields.horizontalVelX_f049_060_compliant.nc"
# title comes from the file name, e.g. "STOFS-3D-Atlantic horizontalVelX_f073_084"
title = "STOFS-3D-Atlantic horizontal Velocity X"
# ----------------------------------------------------------------------------

# The data variable is about 7 GB. With dask installed it is written one time step at a
# time (about 600 MB), so the whole variable is never held in memory. Without dask,
# writing needs one free block of about 7 GB and can fail with a MemoryError.
try:
    import dask
    dask.config.set(scheduler="synchronous")   # one chunk at a time keeps memory low
    chunks = {"time": 1}
except ImportError:
    chunks = None
ds = xr.open_dataset(src, mask_and_scale=False, decode_times=False, chunks=chunks)

# Velocity component names. ASSUMPTION: X/Y are the east/north components
# (geographic frame). Verify against the SCHISM/STOFS documentation; if they
# are model-grid components, drop standard_name and keep long_name only.
VEL = {
    "horizontalVelX": dict(standard_name="eastward_sea_water_velocity",
                           long_name="eastward sea water velocity"),
    "horizontalVelY": dict(standard_name="northward_sea_water_velocity",
                           long_name="northward sea water velocity"),
}
vel_name = next(v for v in VEL if v in ds.variables)

# ===== UGRID (same as the other scripts) =====================================
# mesh variable as a dimensionless int32 scalar, without the optional
# face/edge coordinate references (they conflict with CF's one lon/lat rule)
mesh_attrs = dict(ds["SCHISM_hgrid"].attrs)
mesh_attrs.pop("edge_coordinates", None)
mesh_attrs.pop("face_coordinates", None)
ds = ds.drop_vars("SCHISM_hgrid")
ds["SCHISM_hgrid"] = xr.Variable((), np.int32(0), mesh_attrs)

# connectivity arrays (int32; fill value only on faces)
ds["SCHISM_hgrid_face_nodes"] = ds["SCHISM_hgrid_face_nodes"].astype("int32")
ds["SCHISM_hgrid_edge_nodes"] = ds["SCHISM_hgrid_edge_nodes"].astype("int32")
for v in ["SCHISM_hgrid_face_nodes", "SCHISM_hgrid_edge_nodes"]:
    ds[v].attrs.pop("_FillValue", None)
    ds[v].encoding.pop("_FillValue", None)

# ===== CF: coordinates ========================================================
ds["SCHISM_hgrid_node_x"].attrs.update(standard_name="longitude", long_name="node longitude", units="degrees_east")
ds["SCHISM_hgrid_node_y"].attrs.update(standard_name="latitude", long_name="node latitude", units="degrees_north")
for v in ["SCHISM_hgrid_face_x", "SCHISM_hgrid_face_y", "SCHISM_hgrid_edge_x", "SCHISM_hgrid_edge_y"]:
    if v in ds.variables:
        ds[v].attrs.pop("standard_name", None)
        ds[v].attrs.setdefault("long_name", v.replace("SCHISM_hgrid_", "").replace("_", " ") + " coordinate")

# ===== CF: names and units for THIS file (check against your data) ===========
ds["depth"].attrs.update(standard_name="sea_floor_depth_below_geoid", long_name="bathymetry at nodes", units="m", positive="down")
ds["minimum_depth"].attrs.update(long_name="minimum water depth for wetting/drying", units="m")
ds["bottom_index_node"].attrs.update(long_name="bottom layer index at each node")
ds[vel_name].attrs.update(VEL[vel_name])
ds[vel_name].attrs["units"] = "m s-1"   # overwrites the units NCO set

# UGRID: tell readers which mesh and location each data variable lives on
for v in ["depth", "bottom_index_node", vel_name]:
    ds[v].attrs["mesh"] = "SCHISM_hgrid"
    ds[v].attrs["location"] = "node"

# ===== CF: fill values =======================================================
# missing_value must equal _FillValue: keep a single attribute
for name in ds.variables:
    a = ds[name].attrs
    if "missing_value" in a:
        a.setdefault("_FillValue", a["missing_value"])
        del a["missing_value"]
# coordinate variables must not have _FillValue
ds["time"].attrs.pop("_FillValue", None)
ds["time"].encoding.pop("_FillValue", None)

# CF: time calendar stated explicitly (it is the default)
ds["time"].attrs["calendar"] = "standard"

# ===== CF: global attributes =================================================
ds.attrs["Conventions"] = "CF-1.11 UGRID-1.0"
ds.attrs["title"] = title

ds.to_netcdf(out, encoding={
    "time": {"_FillValue": None},
    "SCHISM_hgrid_edge_nodes": {"_FillValue": None},
    "SCHISM_hgrid_face_nodes": {"dtype": "int32", "_FillValue": np.int32(-1)},
})
print("Wrote", out)

Wrote stofs_3d_atl.t12z.fields.horizontalVelX_f049_060_compliant.nc


**Test.** Compares the new file with the original: values, statistics, fill handling, connectivity, and a read through xugrid.


In [9]:
"""
Check that a *_compliant.nc file still holds the same data as the original:

 1. every numeric variable has identical values (NaN positions included)
 2. statistics (valid count, masked count, sum, mean, min, max) are identical
 3. fill / missing values are understood: the number of masked (NaN) values in
    the decoded new file equals the number of raw _FillValue (+ NaN) values
 4. the connectivity arrays are unchanged
 5. xugrid opens the new file, builds the mesh with the right sizes and
    connectivity, and a xugrid reduction (mean) matches xarray on the original
"""
import sys

import numpy as np
import xarray as xr
import xugrid as xu

# ---- set your files here ---------------------------------------------------
ORIG = "stofs_3d_atl.t12z.fields.horizontalVelX_f049_060.nc"
NEW = "stofs_3d_atl.t12z.fields.horizontalVelX_f049_060_compliant.nc"

# variables that were renamed in the new file (field2d), else leave empty:
# RENAMES = {"uvel4.5": "uvel_4p5m", "vvel4.5": "vvel_4p5m",
#            "uvel0.5": "uvel_0p5m", "vvel0.5": "vvel_0p5m"}
RENAMES = {}

# which time steps to test: None = all (slow on 8 GB files), or e.g. [0, -1]
TIME_STEPS = None
# ----------------------------------------------------------------------------

SKIP = {"SCHISM_hgrid"}   # placeholder variable; its type was changed on purpose
FACE_CANDIDATES = ["SCHISM_hgrid_face_nodes", "element"]
EDGE_CANDIDATES = ["SCHISM_hgrid_edge_nodes"]

results = []


def check(name, ok, msg=""):
    results.append((name, bool(ok)))
    print(("PASS  " if ok else "FAIL  ") + name + (f"   {msg}" if msg else ""))


class Stats:
    def __init__(self):
        self.n = 0          # valid values
        self.nnan = 0       # masked (NaN) values
        self.sum = 0.0
        self.min = np.inf
        self.max = -np.inf

    def add(self, a):
        if a.dtype.kind == "f":
            nan = int(np.isnan(a).sum())
            self.nnan += nan
            valid = a.size - nan
            if valid:
                self.n += valid
                self.sum += float(np.nansum(a, dtype=np.float64))
                self.min = min(self.min, float(np.nanmin(a)))
                self.max = max(self.max, float(np.nanmax(a)))
        else:
            self.n += a.size
            self.sum += float(a.sum(dtype=np.float64))
            self.min = min(self.min, float(a.min()))
            self.max = max(self.max, float(a.max()))

    @property
    def mean(self):
        return self.sum / self.n if self.n else float("nan")

    def same_as(self, o):
        return (self.n, self.nnan, self.sum, self.min, self.max) == \
               (o.n, o.nnan, o.sum, o.min, o.max)


def step_indices(da):
    """Time steps to test for a variable (None = the whole variable at once)."""
    if "time" in da.dims and da.ndim > 1:
        n = da.sizes["time"]
        return list(range(n)) if TIME_STEPS is None else sorted({i % n for i in TIME_STEPS})
    return [None]


def get(da, i):
    return da.values if i is None else da.isel(time=i).values


def as_xr(x):
    return getattr(x, "obj", x)   # unwrap xugrid objects


# ---- open everything --------------------------------------------------------
orig_dec = xr.open_dataset(ORIG, decode_times=False)                       # fills masked to NaN
new_dec = xr.open_dataset(NEW, decode_times=False)
orig_raw = xr.open_dataset(ORIG, mask_and_scale=False, decode_times=False)  # as stored
new_raw = xr.open_dataset(NEW, mask_and_scale=False, decode_times=False)
uds = xu.open_dataset(NEW)

face_name = next(n for n in FACE_CANDIDATES if n in orig_raw.variables)
edge_name = next((n for n in EDGE_CANDIDATES if n in orig_raw.variables), None)
conn_names = {face_name} | ({edge_name} if edge_name else set())

# ---- 1-3. every numeric variable --------------------------------------------
print("\n=== Variables: values, statistics, fill/missing handling ===")
rows = []
for v in orig_dec.variables:
    if v in SKIP or v in conn_names:
        continue
    od = orig_dec[v]
    if od.dtype.kind not in "fiu":
        continue
    rn = RENAMES.get(v, v)
    if rn not in new_dec.variables:
        check(v, False, "missing in new file")
        continue
    nd, nr = new_dec[rn], new_raw[rn]
    if od.dims != nd.dims or od.shape != nd.shape:
        check(v, False, f"dims/shape differ: {od.dims}{od.shape} vs {nd.dims}{nd.shape}")
        continue

    so, sn = Stats(), Stats()
    identical = True
    n_fill_raw = n_nan_raw = 0
    fill = nr.attrs.get("_FillValue")
    for i in step_indices(od):
        a_o, a_n, raw = get(od, i), get(nd, i), get(nr, i)
        identical &= np.array_equal(a_o, a_n, equal_nan=(a_o.dtype.kind == "f"))
        so.add(a_o)
        sn.add(a_n)
        if fill is not None:
            n_fill_raw += int((raw == fill).sum())
        if raw.dtype.kind == "f":
            n_nan_raw += int(np.isnan(raw).sum())

    ok_vals = identical and so.same_as(sn)
    expected_masked = n_fill_raw + n_nan_raw
    ok_fill = sn.nnan == expected_masked
    ok_attr = ("missing_value" not in nr.attrs) or (nr.attrs.get("missing_value") == fill)
    check(f"{v}: values and statistics identical", ok_vals,
          "" if ok_vals else f"orig n={so.n} sum={so.sum} | new n={sn.n} sum={sn.sum}")
    check(f"{v}: fill/missing understood", ok_fill and ok_attr,
          f"masked in decoded new = {sn.nnan}, stored fill/NaN = {expected_masked}, "
          f"_FillValue = {fill}")

    # xugrid reading of the same variable (first tested time step)
    if rn != "time" and rn not in as_xr(uds).variables:
        # xugrid does not expose every variable (e.g. the "crs" grid-mapping
        # variable); the xarray checks above already cover it
        print(f"NOTE  {v}: not exposed by xugrid, xugrid comparison skipped")
    elif rn != "time":
        try:
            i0 = step_indices(od)[0]
            ua = uds[rn]
            xa = as_xr(ua)
            a_u = get(xa, i0)
            res_u = ua if i0 is None else ua.isel(time=i0)
            mean_u = float(as_xr(res_u.mean()))
            od0 = od if i0 is None else od.isel(time=i0)
            mean_o = float(od0.mean())
            ok_u = np.array_equal(a_u, get(od, i0), equal_nan=(a_u.dtype.kind == "f")) \
                and np.isclose(mean_u, mean_o, rtol=1e-6, atol=0, equal_nan=True)
            check(f"{v}: xugrid values and mean match original", ok_u,
                  f"xugrid mean = {mean_u:.6g}, original mean = {mean_o:.6g}")
        except Exception as e:  # report, don't stop
            check(f"{v}: xugrid values and mean match original", False, repr(e))

    rows.append((v, str(od.dtype), sn.n, sn.nnan, sn.mean, sn.min, sn.max))

# ---- 4. connectivity arrays (as stored) -------------------------------------
print("\n=== Connectivity arrays ===")
for name in sorted(conn_names):
    fo = np.nan_to_num(orig_raw[name].values.astype("float64"), nan=-1).astype("int64")
    fn = new_raw[name].values.astype("int64")
    check(f"{name}: unchanged as stored", np.array_equal(fo, fn))

# ---- 5. xugrid mesh ---------------------------------------------------------
print("\n=== xugrid mesh ===")
grid = uds.ugrid.grids[0]
mesh_var = next(n for n, x in new_raw.variables.items()
                if x.attrs.get("cf_role") == "mesh_topology")
node_x_name = new_raw[mesh_var].attrs["node_coordinates"].split()[0]
fn = new_raw[face_name].values.astype("int64")
start = int(new_raw[face_name].attrs.get("start_index", 0))
check("mesh variable present", True, f"'{mesh_var}'")
check("n_node matches", grid.n_node == new_raw[node_x_name].size,
      f"xugrid {grid.n_node}, file {new_raw[node_x_name].size}")
check("n_face matches", grid.n_face == fn.shape[0],
      f"xugrid {grid.n_face}, file {fn.shape[0]}")
expected = np.where(fn >= start, fn - start, -1)   # xugrid uses 0-based indices
gc = np.asarray(grid.face_node_connectivity)
check("face connectivity read correctly by xugrid",
      gc.shape == expected.shape and np.array_equal(gc, expected),
      f"shape {gc.shape} vs expected {expected.shape}")

# ---- summary ----------------------------------------------------------------
print("\n=== Statistics of the new file (decoded; fill values excluded) ===")
print(f"{'variable':28s}{'dtype':9s}{'valid':>14s}{'masked':>12s}{'mean':>14s}{'min':>14s}{'max':>14s}")
for r in rows:
    print(f"{r[0]:28s}{r[1]:9s}{r[2]:14d}{r[3]:12d}{r[4]:14.6g}{r[5]:14.6g}{r[6]:14.6g}")

failed = [n for n, ok in results if not ok]
print(f"\n{len(results) - len(failed)} passed, {len(failed)} failed")
for n in failed:
    print("  FAILED:", n)
sys.exit(1 if failed else 0)


=== Variables: values, statistics, fill/missing handling ===
PASS  minimum_depth: values and statistics identical
PASS  minimum_depth: fill/missing understood   masked in decoded new = 0, stored fill/NaN = 0, _FillValue = nan
PASS  minimum_depth: xugrid values and mean match original   xugrid mean = 0.0001, original mean = 0.0001
PASS  crs: values and statistics identical
PASS  crs: fill/missing understood   masked in decoded new = 0, stored fill/NaN = 0, _FillValue = None
NOTE  crs: not exposed by xugrid, xugrid comparison skipped
PASS  depth: values and statistics identical
PASS  depth: fill/missing understood   masked in decoded new = 0, stored fill/NaN = 0, _FillValue = nan
PASS  depth: xugrid values and mean match original   xugrid mean = 484.262, original mean = 484.262
PASS  bottom_index_node: values and statistics identical
PASS  bottom_index_node: fill/missing understood   masked in decoded new = 0, stored fill/NaN = 0, _FillValue = None
PASS  bottom_index_node: xugrid values

SystemExit: 0

/lustre/env/miniconda3/envs/STOFS-Observer/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## 6. horizontalVelY files

**What will change** (besides the common changes and the shared 3D variables above)

| Variable | Changes |
| --- | --- |
| `horizontalVelY` | `standard_name = northward_sea_water_velocity`, `long_name = northward sea water velocity`, units set explicitly to `m s-1` (assumes the north component) |


In [10]:
import numpy as np
import xarray as xr

# ---- set your files here ---------------------------------------------------
# Works for both horizontalVelX and horizontalVelY files.
src = "stofs_3d_atl.t12z.fields.horizontalVelY_f025_036.nc"
out = "stofs_3d_atl.t12z.fields.horizontalVelY_f025_036_compliant.nc"
# title comes from the file name, e.g. "STOFS-3D-Atlantic horizontalVelX_f073_084"
title = "STOFS-3D-Atlantic horizontal Velocity Y"
# ----------------------------------------------------------------------------

# The data variable is about 7 GB. With dask installed it is written one time step at a
# time (about 600 MB), so the whole variable is never held in memory. Without dask,
# writing needs one free block of about 7 GB and can fail with a MemoryError.
try:
    import dask
    dask.config.set(scheduler="synchronous")   # one chunk at a time keeps memory low
    chunks = {"time": 1}
except ImportError:
    chunks = None
ds = xr.open_dataset(src, mask_and_scale=False, decode_times=False, chunks=chunks)

# Velocity component names. ASSUMPTION: X/Y are the east/north components
# (geographic frame). Verify against the SCHISM/STOFS documentation; if they
# are model-grid components, drop standard_name and keep long_name only.
VEL = {
    "horizontalVelX": dict(standard_name="eastward_sea_water_velocity",
                           long_name="eastward sea water velocity"),
    "horizontalVelY": dict(standard_name="northward_sea_water_velocity",
                           long_name="northward sea water velocity"),
}
vel_name = next(v for v in VEL if v in ds.variables)

# ===== UGRID (same as the other scripts) =====================================
# mesh variable as a dimensionless int32 scalar, without the optional
# face/edge coordinate references (they conflict with CF's one lon/lat rule)
mesh_attrs = dict(ds["SCHISM_hgrid"].attrs)
mesh_attrs.pop("edge_coordinates", None)
mesh_attrs.pop("face_coordinates", None)
ds = ds.drop_vars("SCHISM_hgrid")
ds["SCHISM_hgrid"] = xr.Variable((), np.int32(0), mesh_attrs)

# connectivity arrays (int32; fill value only on faces)
ds["SCHISM_hgrid_face_nodes"] = ds["SCHISM_hgrid_face_nodes"].astype("int32")
ds["SCHISM_hgrid_edge_nodes"] = ds["SCHISM_hgrid_edge_nodes"].astype("int32")
for v in ["SCHISM_hgrid_face_nodes", "SCHISM_hgrid_edge_nodes"]:
    ds[v].attrs.pop("_FillValue", None)
    ds[v].encoding.pop("_FillValue", None)

# ===== CF: coordinates ========================================================
ds["SCHISM_hgrid_node_x"].attrs.update(standard_name="longitude", long_name="node longitude", units="degrees_east")
ds["SCHISM_hgrid_node_y"].attrs.update(standard_name="latitude", long_name="node latitude", units="degrees_north")
for v in ["SCHISM_hgrid_face_x", "SCHISM_hgrid_face_y", "SCHISM_hgrid_edge_x", "SCHISM_hgrid_edge_y"]:
    if v in ds.variables:
        ds[v].attrs.pop("standard_name", None)
        ds[v].attrs.setdefault("long_name", v.replace("SCHISM_hgrid_", "").replace("_", " ") + " coordinate")

# ===== CF: names and units for THIS file (check against your data) ===========
ds["depth"].attrs.update(standard_name="sea_floor_depth_below_geoid", long_name="bathymetry at nodes", units="m", positive="down")
ds["minimum_depth"].attrs.update(long_name="minimum water depth for wetting/drying", units="m")
ds["bottom_index_node"].attrs.update(long_name="bottom layer index at each node")
ds[vel_name].attrs.update(VEL[vel_name])
ds[vel_name].attrs["units"] = "m s-1"   # overwrites the units NCO set

# UGRID: tell readers which mesh and location each data variable lives on
for v in ["depth", "bottom_index_node", vel_name]:
    ds[v].attrs["mesh"] = "SCHISM_hgrid"
    ds[v].attrs["location"] = "node"

# ===== CF: fill values =======================================================
# missing_value must equal _FillValue: keep a single attribute
for name in ds.variables:
    a = ds[name].attrs
    if "missing_value" in a:
        a.setdefault("_FillValue", a["missing_value"])
        del a["missing_value"]
# coordinate variables must not have _FillValue
ds["time"].attrs.pop("_FillValue", None)
ds["time"].encoding.pop("_FillValue", None)

# CF: time calendar stated explicitly (it is the default)
ds["time"].attrs["calendar"] = "standard"

# ===== CF: global attributes =================================================
ds.attrs["Conventions"] = "CF-1.11 UGRID-1.0"
ds.attrs["title"] = title

ds.to_netcdf(out, encoding={
    "time": {"_FillValue": None},
    "SCHISM_hgrid_edge_nodes": {"_FillValue": None},
    "SCHISM_hgrid_face_nodes": {"dtype": "int32", "_FillValue": np.int32(-1)},
})
print("Wrote", out)

Wrote stofs_3d_atl.t12z.fields.horizontalVelY_f025_036_compliant.nc


**Test.** Compares the new file with the original: values, statistics, fill handling, connectivity, and a read through xugrid.


In [11]:
"""
Check that a *_compliant.nc file still holds the same data as the original:

 1. every numeric variable has identical values (NaN positions included)
 2. statistics (valid count, masked count, sum, mean, min, max) are identical
 3. fill / missing values are understood: the number of masked (NaN) values in
    the decoded new file equals the number of raw _FillValue (+ NaN) values
 4. the connectivity arrays are unchanged
 5. xugrid opens the new file, builds the mesh with the right sizes and
    connectivity, and a xugrid reduction (mean) matches xarray on the original
"""
import sys

import numpy as np
import xarray as xr
import xugrid as xu

# ---- set your files here ---------------------------------------------------
ORIG = "stofs_3d_atl.t12z.fields.horizontalVelY_f025_036.nc"
NEW = "stofs_3d_atl.t12z.fields.horizontalVelY_f025_036_compliant.nc"

# variables that were renamed in the new file (field2d), else leave empty:
# RENAMES = {"uvel4.5": "uvel_4p5m", "vvel4.5": "vvel_4p5m",
#            "uvel0.5": "uvel_0p5m", "vvel0.5": "vvel_0p5m"}
RENAMES = {}

# which time steps to test: None = all (slow on 8 GB files), or e.g. [0, -1]
TIME_STEPS = None
# ----------------------------------------------------------------------------

SKIP = {"SCHISM_hgrid"}   # placeholder variable; its type was changed on purpose
FACE_CANDIDATES = ["SCHISM_hgrid_face_nodes", "element"]
EDGE_CANDIDATES = ["SCHISM_hgrid_edge_nodes"]

results = []


def check(name, ok, msg=""):
    results.append((name, bool(ok)))
    print(("PASS  " if ok else "FAIL  ") + name + (f"   {msg}" if msg else ""))


class Stats:
    def __init__(self):
        self.n = 0          # valid values
        self.nnan = 0       # masked (NaN) values
        self.sum = 0.0
        self.min = np.inf
        self.max = -np.inf

    def add(self, a):
        if a.dtype.kind == "f":
            nan = int(np.isnan(a).sum())
            self.nnan += nan
            valid = a.size - nan
            if valid:
                self.n += valid
                self.sum += float(np.nansum(a, dtype=np.float64))
                self.min = min(self.min, float(np.nanmin(a)))
                self.max = max(self.max, float(np.nanmax(a)))
        else:
            self.n += a.size
            self.sum += float(a.sum(dtype=np.float64))
            self.min = min(self.min, float(a.min()))
            self.max = max(self.max, float(a.max()))

    @property
    def mean(self):
        return self.sum / self.n if self.n else float("nan")

    def same_as(self, o):
        return (self.n, self.nnan, self.sum, self.min, self.max) == \
               (o.n, o.nnan, o.sum, o.min, o.max)


def step_indices(da):
    """Time steps to test for a variable (None = the whole variable at once)."""
    if "time" in da.dims and da.ndim > 1:
        n = da.sizes["time"]
        return list(range(n)) if TIME_STEPS is None else sorted({i % n for i in TIME_STEPS})
    return [None]


def get(da, i):
    return da.values if i is None else da.isel(time=i).values


def as_xr(x):
    return getattr(x, "obj", x)   # unwrap xugrid objects


# ---- open everything --------------------------------------------------------
orig_dec = xr.open_dataset(ORIG, decode_times=False)                       # fills masked to NaN
new_dec = xr.open_dataset(NEW, decode_times=False)
orig_raw = xr.open_dataset(ORIG, mask_and_scale=False, decode_times=False)  # as stored
new_raw = xr.open_dataset(NEW, mask_and_scale=False, decode_times=False)
uds = xu.open_dataset(NEW)

face_name = next(n for n in FACE_CANDIDATES if n in orig_raw.variables)
edge_name = next((n for n in EDGE_CANDIDATES if n in orig_raw.variables), None)
conn_names = {face_name} | ({edge_name} if edge_name else set())

# ---- 1-3. every numeric variable --------------------------------------------
print("\n=== Variables: values, statistics, fill/missing handling ===")
rows = []
for v in orig_dec.variables:
    if v in SKIP or v in conn_names:
        continue
    od = orig_dec[v]
    if od.dtype.kind not in "fiu":
        continue
    rn = RENAMES.get(v, v)
    if rn not in new_dec.variables:
        check(v, False, "missing in new file")
        continue
    nd, nr = new_dec[rn], new_raw[rn]
    if od.dims != nd.dims or od.shape != nd.shape:
        check(v, False, f"dims/shape differ: {od.dims}{od.shape} vs {nd.dims}{nd.shape}")
        continue

    so, sn = Stats(), Stats()
    identical = True
    n_fill_raw = n_nan_raw = 0
    fill = nr.attrs.get("_FillValue")
    for i in step_indices(od):
        a_o, a_n, raw = get(od, i), get(nd, i), get(nr, i)
        identical &= np.array_equal(a_o, a_n, equal_nan=(a_o.dtype.kind == "f"))
        so.add(a_o)
        sn.add(a_n)
        if fill is not None:
            n_fill_raw += int((raw == fill).sum())
        if raw.dtype.kind == "f":
            n_nan_raw += int(np.isnan(raw).sum())

    ok_vals = identical and so.same_as(sn)
    expected_masked = n_fill_raw + n_nan_raw
    ok_fill = sn.nnan == expected_masked
    ok_attr = ("missing_value" not in nr.attrs) or (nr.attrs.get("missing_value") == fill)
    check(f"{v}: values and statistics identical", ok_vals,
          "" if ok_vals else f"orig n={so.n} sum={so.sum} | new n={sn.n} sum={sn.sum}")
    check(f"{v}: fill/missing understood", ok_fill and ok_attr,
          f"masked in decoded new = {sn.nnan}, stored fill/NaN = {expected_masked}, "
          f"_FillValue = {fill}")

    # xugrid reading of the same variable (first tested time step)
    if rn != "time" and rn not in as_xr(uds).variables:
        # xugrid does not expose every variable (e.g. the "crs" grid-mapping
        # variable); the xarray checks above already cover it
        print(f"NOTE  {v}: not exposed by xugrid, xugrid comparison skipped")
    elif rn != "time":
        try:
            i0 = step_indices(od)[0]
            ua = uds[rn]
            xa = as_xr(ua)
            a_u = get(xa, i0)
            res_u = ua if i0 is None else ua.isel(time=i0)
            mean_u = float(as_xr(res_u.mean()))
            od0 = od if i0 is None else od.isel(time=i0)
            mean_o = float(od0.mean())
            ok_u = np.array_equal(a_u, get(od, i0), equal_nan=(a_u.dtype.kind == "f")) \
                and np.isclose(mean_u, mean_o, rtol=1e-6, atol=0, equal_nan=True)
            check(f"{v}: xugrid values and mean match original", ok_u,
                  f"xugrid mean = {mean_u:.6g}, original mean = {mean_o:.6g}")
        except Exception as e:  # report, don't stop
            check(f"{v}: xugrid values and mean match original", False, repr(e))

    rows.append((v, str(od.dtype), sn.n, sn.nnan, sn.mean, sn.min, sn.max))

# ---- 4. connectivity arrays (as stored) -------------------------------------
print("\n=== Connectivity arrays ===")
for name in sorted(conn_names):
    fo = np.nan_to_num(orig_raw[name].values.astype("float64"), nan=-1).astype("int64")
    fn = new_raw[name].values.astype("int64")
    check(f"{name}: unchanged as stored", np.array_equal(fo, fn))

# ---- 5. xugrid mesh ---------------------------------------------------------
print("\n=== xugrid mesh ===")
grid = uds.ugrid.grids[0]
mesh_var = next(n for n, x in new_raw.variables.items()
                if x.attrs.get("cf_role") == "mesh_topology")
node_x_name = new_raw[mesh_var].attrs["node_coordinates"].split()[0]
fn = new_raw[face_name].values.astype("int64")
start = int(new_raw[face_name].attrs.get("start_index", 0))
check("mesh variable present", True, f"'{mesh_var}'")
check("n_node matches", grid.n_node == new_raw[node_x_name].size,
      f"xugrid {grid.n_node}, file {new_raw[node_x_name].size}")
check("n_face matches", grid.n_face == fn.shape[0],
      f"xugrid {grid.n_face}, file {fn.shape[0]}")
expected = np.where(fn >= start, fn - start, -1)   # xugrid uses 0-based indices
gc = np.asarray(grid.face_node_connectivity)
check("face connectivity read correctly by xugrid",
      gc.shape == expected.shape and np.array_equal(gc, expected),
      f"shape {gc.shape} vs expected {expected.shape}")

# ---- summary ----------------------------------------------------------------
print("\n=== Statistics of the new file (decoded; fill values excluded) ===")
print(f"{'variable':28s}{'dtype':9s}{'valid':>14s}{'masked':>12s}{'mean':>14s}{'min':>14s}{'max':>14s}")
for r in rows:
    print(f"{r[0]:28s}{r[1]:9s}{r[2]:14d}{r[3]:12d}{r[4]:14.6g}{r[5]:14.6g}{r[6]:14.6g}")

failed = [n for n, ok in results if not ok]
print(f"\n{len(results) - len(failed)} passed, {len(failed)} failed")
for n in failed:
    print("  FAILED:", n)
sys.exit(1 if failed else 0)


=== Variables: values, statistics, fill/missing handling ===
PASS  minimum_depth: values and statistics identical
PASS  minimum_depth: fill/missing understood   masked in decoded new = 0, stored fill/NaN = 0, _FillValue = nan
PASS  minimum_depth: xugrid values and mean match original   xugrid mean = 0.0001, original mean = 0.0001
PASS  crs: values and statistics identical
PASS  crs: fill/missing understood   masked in decoded new = 0, stored fill/NaN = 0, _FillValue = None
NOTE  crs: not exposed by xugrid, xugrid comparison skipped
PASS  depth: values and statistics identical
PASS  depth: fill/missing understood   masked in decoded new = 0, stored fill/NaN = 0, _FillValue = nan
PASS  depth: xugrid values and mean match original   xugrid mean = 484.262, original mean = 484.262
PASS  bottom_index_node: values and statistics identical
PASS  bottom_index_node: fill/missing understood   masked in decoded new = 0, stored fill/NaN = 0, _FillValue = None
PASS  bottom_index_node: xugrid values

SystemExit: 0

/lustre/env/miniconda3/envs/STOFS-Observer/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## 7. diffusivity files

**What will change** (besides the common changes and the shared 3D variables above)

| Variable | Changes |
| --- | --- |
| `diffusivity` | `standard_name = ocean_vertical_momentum_diffusivity`, `long_name = ocean vertical momentum diffusivity (viscosity)`, units `m2/s` changed to `m2 s-1` |


In [12]:
import numpy as np
import xarray as xr

# ---- set your files here ---------------------------------------------------
src = "stofs_3d_atl.t12z.fields.diffusivity_f037_048.nc"
out = "stofs_3d_atl.t12z.fields.diffusivity_f037_048_compliant.nc"
# title comes from the file name, e.g. "STOFS-3D-Atlantic diffusivity_f061_072"
title = "STOFS-3D-Atlantic Diffusivity"
# ----------------------------------------------------------------------------

# The data variable is about 7 GB. With dask installed it is written one time step at a
# time (about 600 MB), so the whole variable is never held in memory. Without dask,
# writing needs one free block of about 7 GB and can fail with a MemoryError.
try:
    import dask
    dask.config.set(scheduler="synchronous")   # one chunk at a time keeps memory low
    chunks = {"time": 1}
except ImportError:
    chunks = None
ds = xr.open_dataset(src, mask_and_scale=False, decode_times=False, chunks=chunks)

# ===== UGRID (same as the other scripts) =====================================
# mesh variable as a dimensionless int32 scalar, without the optional
# face/edge coordinate references (they conflict with CF's one lon/lat rule)
mesh_attrs = dict(ds["SCHISM_hgrid"].attrs)
mesh_attrs.pop("edge_coordinates", None)
mesh_attrs.pop("face_coordinates", None)
ds = ds.drop_vars("SCHISM_hgrid")
ds["SCHISM_hgrid"] = xr.Variable((), np.int32(0), mesh_attrs)

# connectivity arrays (int32; fill value only on faces)
ds["SCHISM_hgrid_face_nodes"] = ds["SCHISM_hgrid_face_nodes"].astype("int32")
ds["SCHISM_hgrid_edge_nodes"] = ds["SCHISM_hgrid_edge_nodes"].astype("int32")
for v in ["SCHISM_hgrid_face_nodes", "SCHISM_hgrid_edge_nodes"]:
    ds[v].attrs.pop("_FillValue", None)
    ds[v].encoding.pop("_FillValue", None)

# ===== CF: coordinates ========================================================
ds["SCHISM_hgrid_node_x"].attrs.update(standard_name="longitude", long_name="node longitude", units="degrees_east")
ds["SCHISM_hgrid_node_y"].attrs.update(standard_name="latitude", long_name="node latitude", units="degrees_north")
for v in ["SCHISM_hgrid_face_x", "SCHISM_hgrid_face_y", "SCHISM_hgrid_edge_x", "SCHISM_hgrid_edge_y"]:
    if v in ds.variables:
        ds[v].attrs.pop("standard_name", None)
        ds[v].attrs.setdefault("long_name", v.replace("SCHISM_hgrid_", "").replace("_", " ") + " coordinate")

# ===== CF: names and units for THIS file (check against your data) ===========
ds["depth"].attrs.update(standard_name="sea_floor_depth_below_geoid", long_name="bathymetry at nodes", units="m", positive="down")
ds["minimum_depth"].attrs.update(long_name="minimum water depth for wetting/drying", units="m")
ds["bottom_index_node"].attrs.update(long_name="bottom layer index at each node")
# Confirmed by a colleague: the "diffusivity" variable is the ocean vertical
# momentum diffusivity (viscosity), in m2 s-1.
ds["diffusivity"].attrs.update(
    standard_name="ocean_vertical_momentum_diffusivity",
    long_name="ocean vertical momentum diffusivity (viscosity)",
    units="m2 s-1",     # normalizes the "m2/s" string NCO set
)

# UGRID: tell readers which mesh and location each data variable lives on
for v in ["depth", "bottom_index_node", "diffusivity"]:
    ds[v].attrs["mesh"] = "SCHISM_hgrid"
    ds[v].attrs["location"] = "node"

# ===== CF: fill values =======================================================
# missing_value must equal _FillValue: keep a single attribute
for name in ds.variables:
    a = ds[name].attrs
    if "missing_value" in a:
        a.setdefault("_FillValue", a["missing_value"])
        del a["missing_value"]
# coordinate variables must not have _FillValue
ds["time"].attrs.pop("_FillValue", None)
ds["time"].encoding.pop("_FillValue", None)

# CF: time calendar stated explicitly (it is the default)
ds["time"].attrs["calendar"] = "standard"

# ===== CF: global attributes =================================================
ds.attrs["Conventions"] = "CF-1.11 UGRID-1.0"
ds.attrs["title"] = title

ds.to_netcdf(out, encoding={
    "time": {"_FillValue": None},
    "SCHISM_hgrid_edge_nodes": {"_FillValue": None},
    "SCHISM_hgrid_face_nodes": {"dtype": "int32", "_FillValue": np.int32(-1)},
})
print("Wrote", out)

Wrote stofs_3d_atl.t12z.fields.diffusivity_f037_048_compliant.nc


**Test.** Compares the new file with the original: values, statistics, fill handling, connectivity, and a read through xugrid.


In [13]:
"""
Check that a *_compliant.nc file still holds the same data as the original:

 1. every numeric variable has identical values (NaN positions included)
 2. statistics (valid count, masked count, sum, mean, min, max) are identical
 3. fill / missing values are understood: the number of masked (NaN) values in
    the decoded new file equals the number of raw _FillValue (+ NaN) values
 4. the connectivity arrays are unchanged
 5. xugrid opens the new file, builds the mesh with the right sizes and
    connectivity, and a xugrid reduction (mean) matches xarray on the original
"""
import sys

import numpy as np
import xarray as xr
import xugrid as xu

# ---- set your files here ---------------------------------------------------
ORIG = "stofs_3d_atl.t12z.fields.diffusivity_f037_048.nc"
NEW = "stofs_3d_atl.t12z.fields.diffusivity_f037_048_compliant.nc"

# variables that were renamed in the new file (field2d), else leave empty:
# RENAMES = {"uvel4.5": "uvel_4p5m", "vvel4.5": "vvel_4p5m",
#            "uvel0.5": "uvel_0p5m", "vvel0.5": "vvel_0p5m"}
RENAMES = {}

# which time steps to test: None = all (slow on 8 GB files), or e.g. [0, -1]
TIME_STEPS = None
# ----------------------------------------------------------------------------

SKIP = {"SCHISM_hgrid"}   # placeholder variable; its type was changed on purpose
FACE_CANDIDATES = ["SCHISM_hgrid_face_nodes", "element"]
EDGE_CANDIDATES = ["SCHISM_hgrid_edge_nodes"]

results = []


def check(name, ok, msg=""):
    results.append((name, bool(ok)))
    print(("PASS  " if ok else "FAIL  ") + name + (f"   {msg}" if msg else ""))


class Stats:
    def __init__(self):
        self.n = 0          # valid values
        self.nnan = 0       # masked (NaN) values
        self.sum = 0.0
        self.min = np.inf
        self.max = -np.inf

    def add(self, a):
        if a.dtype.kind == "f":
            nan = int(np.isnan(a).sum())
            self.nnan += nan
            valid = a.size - nan
            if valid:
                self.n += valid
                self.sum += float(np.nansum(a, dtype=np.float64))
                self.min = min(self.min, float(np.nanmin(a)))
                self.max = max(self.max, float(np.nanmax(a)))
        else:
            self.n += a.size
            self.sum += float(a.sum(dtype=np.float64))
            self.min = min(self.min, float(a.min()))
            self.max = max(self.max, float(a.max()))

    @property
    def mean(self):
        return self.sum / self.n if self.n else float("nan")

    def same_as(self, o):
        return (self.n, self.nnan, self.sum, self.min, self.max) == \
               (o.n, o.nnan, o.sum, o.min, o.max)


def step_indices(da):
    """Time steps to test for a variable (None = the whole variable at once)."""
    if "time" in da.dims and da.ndim > 1:
        n = da.sizes["time"]
        return list(range(n)) if TIME_STEPS is None else sorted({i % n for i in TIME_STEPS})
    return [None]


def get(da, i):
    return da.values if i is None else da.isel(time=i).values


def as_xr(x):
    return getattr(x, "obj", x)   # unwrap xugrid objects


# ---- open everything --------------------------------------------------------
orig_dec = xr.open_dataset(ORIG, decode_times=False)                       # fills masked to NaN
new_dec = xr.open_dataset(NEW, decode_times=False)
orig_raw = xr.open_dataset(ORIG, mask_and_scale=False, decode_times=False)  # as stored
new_raw = xr.open_dataset(NEW, mask_and_scale=False, decode_times=False)
uds = xu.open_dataset(NEW)

face_name = next(n for n in FACE_CANDIDATES if n in orig_raw.variables)
edge_name = next((n for n in EDGE_CANDIDATES if n in orig_raw.variables), None)
conn_names = {face_name} | ({edge_name} if edge_name else set())

# ---- 1-3. every numeric variable --------------------------------------------
print("\n=== Variables: values, statistics, fill/missing handling ===")
rows = []
for v in orig_dec.variables:
    if v in SKIP or v in conn_names:
        continue
    od = orig_dec[v]
    if od.dtype.kind not in "fiu":
        continue
    rn = RENAMES.get(v, v)
    if rn not in new_dec.variables:
        check(v, False, "missing in new file")
        continue
    nd, nr = new_dec[rn], new_raw[rn]
    if od.dims != nd.dims or od.shape != nd.shape:
        check(v, False, f"dims/shape differ: {od.dims}{od.shape} vs {nd.dims}{nd.shape}")
        continue

    so, sn = Stats(), Stats()
    identical = True
    n_fill_raw = n_nan_raw = 0
    fill = nr.attrs.get("_FillValue")
    for i in step_indices(od):
        a_o, a_n, raw = get(od, i), get(nd, i), get(nr, i)
        identical &= np.array_equal(a_o, a_n, equal_nan=(a_o.dtype.kind == "f"))
        so.add(a_o)
        sn.add(a_n)
        if fill is not None:
            n_fill_raw += int((raw == fill).sum())
        if raw.dtype.kind == "f":
            n_nan_raw += int(np.isnan(raw).sum())

    ok_vals = identical and so.same_as(sn)
    expected_masked = n_fill_raw + n_nan_raw
    ok_fill = sn.nnan == expected_masked
    ok_attr = ("missing_value" not in nr.attrs) or (nr.attrs.get("missing_value") == fill)
    check(f"{v}: values and statistics identical", ok_vals,
          "" if ok_vals else f"orig n={so.n} sum={so.sum} | new n={sn.n} sum={sn.sum}")
    check(f"{v}: fill/missing understood", ok_fill and ok_attr,
          f"masked in decoded new = {sn.nnan}, stored fill/NaN = {expected_masked}, "
          f"_FillValue = {fill}")

    # xugrid reading of the same variable (first tested time step)
    if rn != "time" and rn not in as_xr(uds).variables:
        # xugrid does not expose every variable (e.g. the "crs" grid-mapping
        # variable); the xarray checks above already cover it
        print(f"NOTE  {v}: not exposed by xugrid, xugrid comparison skipped")
    elif rn != "time":
        try:
            i0 = step_indices(od)[0]
            ua = uds[rn]
            xa = as_xr(ua)
            a_u = get(xa, i0)
            res_u = ua if i0 is None else ua.isel(time=i0)
            mean_u = float(as_xr(res_u.mean()))
            od0 = od if i0 is None else od.isel(time=i0)
            mean_o = float(od0.mean())
            ok_u = np.array_equal(a_u, get(od, i0), equal_nan=(a_u.dtype.kind == "f")) \
                and np.isclose(mean_u, mean_o, rtol=1e-6, atol=0, equal_nan=True)
            check(f"{v}: xugrid values and mean match original", ok_u,
                  f"xugrid mean = {mean_u:.6g}, original mean = {mean_o:.6g}")
        except Exception as e:  # report, don't stop
            check(f"{v}: xugrid values and mean match original", False, repr(e))

    rows.append((v, str(od.dtype), sn.n, sn.nnan, sn.mean, sn.min, sn.max))

# ---- 4. connectivity arrays (as stored) -------------------------------------
print("\n=== Connectivity arrays ===")
for name in sorted(conn_names):
    fo = np.nan_to_num(orig_raw[name].values.astype("float64"), nan=-1).astype("int64")
    fn = new_raw[name].values.astype("int64")
    check(f"{name}: unchanged as stored", np.array_equal(fo, fn))

# ---- 5. xugrid mesh ---------------------------------------------------------
print("\n=== xugrid mesh ===")
grid = uds.ugrid.grids[0]
mesh_var = next(n for n, x in new_raw.variables.items()
                if x.attrs.get("cf_role") == "mesh_topology")
node_x_name = new_raw[mesh_var].attrs["node_coordinates"].split()[0]
fn = new_raw[face_name].values.astype("int64")
start = int(new_raw[face_name].attrs.get("start_index", 0))
check("mesh variable present", True, f"'{mesh_var}'")
check("n_node matches", grid.n_node == new_raw[node_x_name].size,
      f"xugrid {grid.n_node}, file {new_raw[node_x_name].size}")
check("n_face matches", grid.n_face == fn.shape[0],
      f"xugrid {grid.n_face}, file {fn.shape[0]}")
expected = np.where(fn >= start, fn - start, -1)   # xugrid uses 0-based indices
gc = np.asarray(grid.face_node_connectivity)
check("face connectivity read correctly by xugrid",
      gc.shape == expected.shape and np.array_equal(gc, expected),
      f"shape {gc.shape} vs expected {expected.shape}")

# ---- summary ----------------------------------------------------------------
print("\n=== Statistics of the new file (decoded; fill values excluded) ===")
print(f"{'variable':28s}{'dtype':9s}{'valid':>14s}{'masked':>12s}{'mean':>14s}{'min':>14s}{'max':>14s}")
for r in rows:
    print(f"{r[0]:28s}{r[1]:9s}{r[2]:14d}{r[3]:12d}{r[4]:14.6g}{r[5]:14.6g}{r[6]:14.6g}")

failed = [n for n, ok in results if not ok]
print(f"\n{len(results) - len(failed)} passed, {len(failed)} failed")
for n in failed:
    print("  FAILED:", n)
sys.exit(1 if failed else 0)


=== Variables: values, statistics, fill/missing handling ===
PASS  minimum_depth: values and statistics identical
PASS  minimum_depth: fill/missing understood   masked in decoded new = 0, stored fill/NaN = 0, _FillValue = nan
PASS  minimum_depth: xugrid values and mean match original   xugrid mean = 0.0001, original mean = 0.0001
PASS  crs: values and statistics identical
PASS  crs: fill/missing understood   masked in decoded new = 0, stored fill/NaN = 0, _FillValue = None
NOTE  crs: not exposed by xugrid, xugrid comparison skipped
PASS  depth: values and statistics identical
PASS  depth: fill/missing understood   masked in decoded new = 0, stored fill/NaN = 0, _FillValue = nan
PASS  depth: xugrid values and mean match original   xugrid mean = 484.262, original mean = 484.262
PASS  bottom_index_node: values and statistics identical
PASS  bottom_index_node: fill/missing understood   masked in decoded new = 0, stored fill/NaN = 0, _FillValue = None
PASS  bottom_index_node: xugrid values

SystemExit: 0

/lustre/env/miniconda3/envs/STOFS-Observer/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## 8. out2d files

**What will change.** The 2D fields. `depth`, `minimum_depth` and `bottom_index_node` change as in the 3D files.

| Variable | Changes |
| --- | --- |
| `elevation` | `standard_name = sea_surface_height_above_geoid`; existing `long_name` and units `m` kept; `missing_value = -99999` converted to `_FillValue` |
| `airPressure` | `standard_name = air_pressure_at_mean_sea_level`, `long_name = air pressure at mean sea level`, units `Pa` (confirmed by the values, about 101,500) |
| `windSpeedX`, `windSpeedY` | `eastward_wind`, `northward_wind` (assumes east and north components), units `m/s` changed to `m s-1` |
| `depthAverageVelX`, `depthAverageVelY` | `barotropic_eastward_sea_water_velocity`, `barotropic_northward_sea_water_velocity`, units `m s-1` added |
| `evaporationRate`, `precipitationRate` | `water_evaporation_flux`, `precipitation_flux`, units `kg m-2 s-1` (inferred from the magnitudes; the evaporation sign is unconfirmed) |
| `windStressX`, `windStressY` | `long_name` and units `m2 s-2` only (inferred); no standard name, because CF's stress names are in Pa |
| `dryFlagNode`, `dryFlagElement`, `dryFlagSide` | `long_name`, `flag_values = 0, 1`, `flag_meanings = "wet dry"` (0 is wet, 1 is dry; no CF standard name fits) |
| `isolatedPondNode` | `flag_meanings` rewritten as space-separated words; `flag_values` stored as `int8` |
| `vgrid_dummy` | `long_name` only (a placeholder that defines the vertical layer dimension) |

The next cell prints magnitudes that were used to infer the units of the rate and stress variables. Run it on your own file if you want to check them.


In [15]:
import xarray as xr

filename = "stofs_3d_atl.t12z.fields.out2d_f001_012.nc"   # the original out2d file
d = xr.open_dataset(filename, mask_and_scale=False, decode_times=False)
for v in ["airPressure", "windStressX", "evaporationRate", "precipitationRate"]:
    x = d[v].isel(time=0).values
    print(v, float(abs(x).max()), float(abs(x).mean()))

airPressure 102408.0 101522.9453125
windStressX 0.0013827700167894363 1.0701293831516523e-05
evaporationRate 0.00035966429277323186 1.4383240340976045e-05
precipitationRate 0.04819747433066368 2.0411704099387862e-05


In [14]:
import numpy as np
import xarray as xr

# ---- set your files here ---------------------------------------------------
src = "stofs_3d_atl.t12z.fields.out2d_f001_012.nc"
out = "stofs_3d_atl.t12z.fields.out2d_f001_012_compliant.nc"
# title comes from the file name, e.g. "STOFS-3D-Atlantic out2d_f001_012"
title = "STOFS-3D-Atlantic " + src.split(".")[-2]

# ASSUMPTION switch: set True only if the X/Y vector components (wind,
# depth-averaged velocity) are written as EAST/NORTH components. If they are
# model-grid components, set False: only long_name/units are written then.
ASSUME_EAST_NORTH = True
# ----------------------------------------------------------------------------

ds = xr.open_dataset(src, mask_and_scale=False, decode_times=False)

# ===== UGRID (same as the other scripts) =====================================
# mesh variable as a dimensionless int32 scalar, without the optional
# face/edge coordinate references (they conflict with CF's one lon/lat rule)
mesh_attrs = dict(ds["SCHISM_hgrid"].attrs)
mesh_attrs.pop("edge_coordinates", None)
mesh_attrs.pop("face_coordinates", None)
ds = ds.drop_vars("SCHISM_hgrid")
ds["SCHISM_hgrid"] = xr.Variable((), np.int32(0), mesh_attrs)

# connectivity arrays (int32; fill value only on faces)
ds["SCHISM_hgrid_face_nodes"] = ds["SCHISM_hgrid_face_nodes"].astype("int32")
ds["SCHISM_hgrid_edge_nodes"] = ds["SCHISM_hgrid_edge_nodes"].astype("int32")
for v in ["SCHISM_hgrid_face_nodes", "SCHISM_hgrid_edge_nodes"]:
    ds[v].attrs.pop("_FillValue", None)
    ds[v].encoding.pop("_FillValue", None)

# ===== CF: coordinates ========================================================
ds["SCHISM_hgrid_node_x"].attrs.update(standard_name="longitude", long_name="node longitude", units="degrees_east")
ds["SCHISM_hgrid_node_y"].attrs.update(standard_name="latitude", long_name="node latitude", units="degrees_north")
for v in ["SCHISM_hgrid_face_x", "SCHISM_hgrid_face_y", "SCHISM_hgrid_edge_x", "SCHISM_hgrid_edge_y"]:
    if v in ds.variables:
        ds[v].attrs.pop("standard_name", None)
        ds[v].attrs.setdefault("long_name", v.replace("SCHISM_hgrid_", "").replace("_", " ") + " coordinate")

# ===== CF: names and units for THIS file (check against your data) ===========
META = {
    "depth": dict(standard_name="sea_floor_depth_below_geoid", long_name="bathymetry at nodes", units="m", positive="down"),
    "minimum_depth": dict(long_name="minimum water depth for wetting/drying", units="m"),
    "bottom_index_node": dict(long_name="bottom layer index at each node"),
    "vgrid_dummy": dict(long_name="placeholder (all zeros) that defines the vertical layer dimension"),
    # the long_name already says "above xgeoid20b", a geoid model
    "elevation": dict(standard_name="sea_surface_height_above_geoid"),
    "dryFlagNode": dict(long_name="dry flag at nodes"),
    "dryFlagElement": dict(long_name="dry flag at elements (faces)"),
    "dryFlagSide": dict(long_name="dry flag at sides (edges)"),
    # ASSUMPTION: Pa. Verify by magnitude (values near 1e5 mean Pa).
    "airPressure": dict(standard_name="air_pressure_at_mean_sea_level",
                        long_name="air pressure at mean sea level", units="Pa"),
    # INFERRED FROM MAGNITUDES (not from documentation): mass fluxes in
    # kg m-2 s-1 and kinematic wind stress in m2 s-2. Confirm in the SCHISM/STOFS
    # documentation. Also confirm the sign of evaporationRate (the CF name is
    # positive upward); if it is the opposite, keep only the long_name.
    "evaporationRate": dict(standard_name="water_evaporation_flux",
                            long_name="evaporation rate (mass flux)", units="kg m-2 s-1"),
    "precipitationRate": dict(standard_name="precipitation_flux",
                              long_name="precipitation rate (mass flux)", units="kg m-2 s-1"),
    # no CF standard_name: CF's stress names are in Pa, these are kinematic
    "windStressX": dict(long_name="kinematic wind stress, x component", units="m2 s-2"),
    "windStressY": dict(long_name="kinematic wind stress, y component", units="m2 s-2"),
    "windSpeedX": dict(long_name="wind speed, x component", units="m s-1"),
    "windSpeedY": dict(long_name="wind speed, y component", units="m s-1"),
    "depthAverageVelX": dict(long_name="depth-averaged velocity, x component", units="m s-1"),
    "depthAverageVelY": dict(long_name="depth-averaged velocity, y component", units="m s-1"),
}
# frame-dependent standard names, applied only if ASSUME_EAST_NORTH is True
EAST_NORTH = {
    "windSpeedX": dict(standard_name="eastward_wind", long_name="eastward wind speed"),
    "windSpeedY": dict(standard_name="northward_wind", long_name="northward wind speed"),
    "depthAverageVelX": dict(standard_name="barotropic_eastward_sea_water_velocity",
                             long_name="depth-averaged eastward sea water velocity"),
    "depthAverageVelY": dict(standard_name="barotropic_northward_sea_water_velocity",
                             long_name="depth-averaged northward sea water velocity"),
}
for name, attrs in META.items():
    if name in ds.variables:
        ds[name].attrs.update(attrs)
if ASSUME_EAST_NORTH:
    for name, attrs in EAST_NORTH.items():
        if name in ds.variables:
            ds[name].attrs.update(attrs)

# CF 3.5: flag_meanings must be space-separated words, one per flag value
ds["isolatedPondNode"].attrs["flag_values"] = np.array([0, 1], dtype="int8")
ds["isolatedPondNode"].attrs["flag_meanings"] = "not_isolated_pond_node isolated_pond_node"

# CF 3.5: the dry flags are 0/1 flags (0 = wet, 1 = dry; confirmed by a colleague,
# assumed to apply to all three). flag_values must have the variable's own type.
for v in ("dryFlagNode", "dryFlagElement", "dryFlagSide"):
    if v in ds.variables:
        ds[v].attrs["flag_values"] = np.array([0, 1], dtype=ds[v].dtype)
        ds[v].attrs["flag_meanings"] = "wet dry"

# UGRID: make sure every mesh data variable has mesh and location
STRUCTURAL = {"SCHISM_hgrid", "SCHISM_hgrid_face_nodes", "SCHISM_hgrid_edge_nodes"}
LOCATION_BY_DIM = {"nSCHISM_hgrid_node": "node", "nSCHISM_hgrid_face": "face", "nSCHISM_hgrid_edge": "edge"}
for name in ds.data_vars:
    if name in STRUCTURAL:
        continue
    for dim, loc in LOCATION_BY_DIM.items():
        if dim in ds[name].dims:
            ds[name].attrs.setdefault("mesh", "SCHISM_hgrid")
            ds[name].attrs.setdefault("location", loc)
            break

# ===== CF: fill values =======================================================
# missing_value must equal _FillValue: keep a single attribute
# (elevation has missing_value = -99999 and becomes _FillValue = -99999)
for name in ds.variables:
    a = ds[name].attrs
    if "missing_value" in a:
        a.setdefault("_FillValue", a["missing_value"])
        del a["missing_value"]
# coordinate variables must not have _FillValue
ds["time"].attrs.pop("_FillValue", None)
ds["time"].encoding.pop("_FillValue", None)

# CF: time calendar stated explicitly (it is the default)
ds["time"].attrs["calendar"] = "standard"

# ===== CF: global attributes =================================================
ds.attrs["Conventions"] = "CF-1.11 UGRID-1.0"
ds.attrs["title"] = title

ds.to_netcdf(out, encoding={
    "time": {"_FillValue": None},
    "SCHISM_hgrid_edge_nodes": {"_FillValue": None},
    "SCHISM_hgrid_face_nodes": {"dtype": "int32", "_FillValue": np.int32(-1)},
})
print("Wrote", out)

Wrote stofs_3d_atl.t12z.fields.out2d_f001_012_compliant.nc


**Test.** Compares the new file with the original: values, statistics, fill handling, connectivity, and a read through xugrid.


In [15]:
"""
Check that a *_compliant.nc file still holds the same data as the original:

 1. every numeric variable has identical values (NaN positions included)
 2. statistics (valid count, masked count, sum, mean, min, max) are identical
 3. fill / missing values are understood: the number of masked (NaN) values in
    the decoded new file equals the number of raw _FillValue (+ NaN) values
 4. the connectivity arrays are unchanged
 5. xugrid opens the new file, builds the mesh with the right sizes and
    connectivity, and a xugrid reduction (mean) matches xarray on the original
"""
import sys

import numpy as np
import xarray as xr
import xugrid as xu

# ---- set your files here ---------------------------------------------------
ORIG = "stofs_3d_atl.t12z.fields.out2d_f001_012.nc"
NEW = "stofs_3d_atl.t12z.fields.out2d_f001_012_compliant.nc"

# variables that were renamed in the new file (field2d), else leave empty:
# RENAMES = {"uvel4.5": "uvel_4p5m", "vvel4.5": "vvel_4p5m",
#            "uvel0.5": "uvel_0p5m", "vvel0.5": "vvel_0p5m"}
RENAMES = {}

# which time steps to test: None = all (slow on 8 GB files), or e.g. [0, -1]
TIME_STEPS = None
# ----------------------------------------------------------------------------

SKIP = {"SCHISM_hgrid"}   # placeholder variable; its type was changed on purpose
FACE_CANDIDATES = ["SCHISM_hgrid_face_nodes", "element"]
EDGE_CANDIDATES = ["SCHISM_hgrid_edge_nodes"]

results = []


def check(name, ok, msg=""):
    results.append((name, bool(ok)))
    print(("PASS  " if ok else "FAIL  ") + name + (f"   {msg}" if msg else ""))


class Stats:
    def __init__(self):
        self.n = 0          # valid values
        self.nnan = 0       # masked (NaN) values
        self.sum = 0.0
        self.min = np.inf
        self.max = -np.inf

    def add(self, a):
        if a.dtype.kind == "f":
            nan = int(np.isnan(a).sum())
            self.nnan += nan
            valid = a.size - nan
            if valid:
                self.n += valid
                self.sum += float(np.nansum(a, dtype=np.float64))
                self.min = min(self.min, float(np.nanmin(a)))
                self.max = max(self.max, float(np.nanmax(a)))
        else:
            self.n += a.size
            self.sum += float(a.sum(dtype=np.float64))
            self.min = min(self.min, float(a.min()))
            self.max = max(self.max, float(a.max()))

    @property
    def mean(self):
        return self.sum / self.n if self.n else float("nan")

    def same_as(self, o):
        return (self.n, self.nnan, self.sum, self.min, self.max) == \
               (o.n, o.nnan, o.sum, o.min, o.max)


def step_indices(da):
    """Time steps to test for a variable (None = the whole variable at once)."""
    if "time" in da.dims and da.ndim > 1:
        n = da.sizes["time"]
        return list(range(n)) if TIME_STEPS is None else sorted({i % n for i in TIME_STEPS})
    return [None]


def get(da, i):
    return da.values if i is None else da.isel(time=i).values


def as_xr(x):
    return getattr(x, "obj", x)   # unwrap xugrid objects


# ---- open everything --------------------------------------------------------
orig_dec = xr.open_dataset(ORIG, decode_times=False)                       # fills masked to NaN
new_dec = xr.open_dataset(NEW, decode_times=False)
orig_raw = xr.open_dataset(ORIG, mask_and_scale=False, decode_times=False)  # as stored
new_raw = xr.open_dataset(NEW, mask_and_scale=False, decode_times=False)
uds = xu.open_dataset(NEW)

face_name = next(n for n in FACE_CANDIDATES if n in orig_raw.variables)
edge_name = next((n for n in EDGE_CANDIDATES if n in orig_raw.variables), None)
conn_names = {face_name} | ({edge_name} if edge_name else set())

# ---- 1-3. every numeric variable --------------------------------------------
print("\n=== Variables: values, statistics, fill/missing handling ===")
rows = []
for v in orig_dec.variables:
    if v in SKIP or v in conn_names:
        continue
    od = orig_dec[v]
    if od.dtype.kind not in "fiu":
        continue
    rn = RENAMES.get(v, v)
    if rn not in new_dec.variables:
        check(v, False, "missing in new file")
        continue
    nd, nr = new_dec[rn], new_raw[rn]
    if od.dims != nd.dims or od.shape != nd.shape:
        check(v, False, f"dims/shape differ: {od.dims}{od.shape} vs {nd.dims}{nd.shape}")
        continue

    so, sn = Stats(), Stats()
    identical = True
    n_fill_raw = n_nan_raw = 0
    fill = nr.attrs.get("_FillValue")
    for i in step_indices(od):
        a_o, a_n, raw = get(od, i), get(nd, i), get(nr, i)
        identical &= np.array_equal(a_o, a_n, equal_nan=(a_o.dtype.kind == "f"))
        so.add(a_o)
        sn.add(a_n)
        if fill is not None:
            n_fill_raw += int((raw == fill).sum())
        if raw.dtype.kind == "f":
            n_nan_raw += int(np.isnan(raw).sum())

    ok_vals = identical and so.same_as(sn)
    expected_masked = n_fill_raw + n_nan_raw
    ok_fill = sn.nnan == expected_masked
    ok_attr = ("missing_value" not in nr.attrs) or (nr.attrs.get("missing_value") == fill)
    check(f"{v}: values and statistics identical", ok_vals,
          "" if ok_vals else f"orig n={so.n} sum={so.sum} | new n={sn.n} sum={sn.sum}")
    check(f"{v}: fill/missing understood", ok_fill and ok_attr,
          f"masked in decoded new = {sn.nnan}, stored fill/NaN = {expected_masked}, "
          f"_FillValue = {fill}")

    # xugrid reading of the same variable (first tested time step)
    if rn != "time" and rn not in as_xr(uds).variables:
        # xugrid does not expose every variable (e.g. the "crs" grid-mapping
        # variable); the xarray checks above already cover it
        print(f"NOTE  {v}: not exposed by xugrid, xugrid comparison skipped")
    elif rn != "time":
        try:
            i0 = step_indices(od)[0]
            ua = uds[rn]
            xa = as_xr(ua)
            a_u = get(xa, i0)
            res_u = ua if i0 is None else ua.isel(time=i0)
            mean_u = float(as_xr(res_u.mean()))
            od0 = od if i0 is None else od.isel(time=i0)
            mean_o = float(od0.mean())
            ok_u = np.array_equal(a_u, get(od, i0), equal_nan=(a_u.dtype.kind == "f")) \
                and np.isclose(mean_u, mean_o, rtol=1e-6, atol=0, equal_nan=True)
            check(f"{v}: xugrid values and mean match original", ok_u,
                  f"xugrid mean = {mean_u:.6g}, original mean = {mean_o:.6g}")
        except Exception as e:  # report, don't stop
            check(f"{v}: xugrid values and mean match original", False, repr(e))

    rows.append((v, str(od.dtype), sn.n, sn.nnan, sn.mean, sn.min, sn.max))

# ---- 4. connectivity arrays (as stored) -------------------------------------
print("\n=== Connectivity arrays ===")
for name in sorted(conn_names):
    fo = np.nan_to_num(orig_raw[name].values.astype("float64"), nan=-1).astype("int64")
    fn = new_raw[name].values.astype("int64")
    check(f"{name}: unchanged as stored", np.array_equal(fo, fn))

# ---- 5. xugrid mesh ---------------------------------------------------------
print("\n=== xugrid mesh ===")
grid = uds.ugrid.grids[0]
mesh_var = next(n for n, x in new_raw.variables.items()
                if x.attrs.get("cf_role") == "mesh_topology")
node_x_name = new_raw[mesh_var].attrs["node_coordinates"].split()[0]
fn = new_raw[face_name].values.astype("int64")
start = int(new_raw[face_name].attrs.get("start_index", 0))
check("mesh variable present", True, f"'{mesh_var}'")
check("n_node matches", grid.n_node == new_raw[node_x_name].size,
      f"xugrid {grid.n_node}, file {new_raw[node_x_name].size}")
check("n_face matches", grid.n_face == fn.shape[0],
      f"xugrid {grid.n_face}, file {fn.shape[0]}")
expected = np.where(fn >= start, fn - start, -1)   # xugrid uses 0-based indices
gc = np.asarray(grid.face_node_connectivity)
check("face connectivity read correctly by xugrid",
      gc.shape == expected.shape and np.array_equal(gc, expected),
      f"shape {gc.shape} vs expected {expected.shape}")

# ---- summary ----------------------------------------------------------------
print("\n=== Statistics of the new file (decoded; fill values excluded) ===")
print(f"{'variable':28s}{'dtype':9s}{'valid':>14s}{'masked':>12s}{'mean':>14s}{'min':>14s}{'max':>14s}")
for r in rows:
    print(f"{r[0]:28s}{r[1]:9s}{r[2]:14d}{r[3]:12d}{r[4]:14.6g}{r[5]:14.6g}{r[6]:14.6g}")

failed = [n for n, ok in results if not ok]
print(f"\n{len(results) - len(failed)} passed, {len(failed)} failed")
for n in failed:
    print("  FAILED:", n)
sys.exit(1 if failed else 0)


=== Variables: values, statistics, fill/missing handling ===
PASS  elevation: values and statistics identical
PASS  elevation: fill/missing understood   masked in decoded new = 2394636, stored fill/NaN = 2394636, _FillValue = -99999.0
PASS  elevation: xugrid values and mean match original   xugrid mean = 1.76043, original mean = 1.76043
PASS  vgrid_dummy: values and statistics identical
PASS  vgrid_dummy: fill/missing understood   masked in decoded new = 0, stored fill/NaN = 0, _FillValue = nan
PASS  vgrid_dummy: xugrid values and mean match original   xugrid mean = 0, original mean = 0
PASS  minimum_depth: values and statistics identical
PASS  minimum_depth: fill/missing understood   masked in decoded new = 0, stored fill/NaN = 0, _FillValue = nan
PASS  minimum_depth: xugrid values and mean match original   xugrid mean = 0.0001, original mean = 0.0001
PASS  crs: values and statistics identical
PASS  crs: fill/missing understood   masked in decoded new = 0, stored fill/NaN = 0, _FillV

SystemExit: 0

/lustre/env/miniconda3/envs/STOFS-Observer/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## 9. field2d files

**What will change.** The native SCHISM 2D output has a different structure from the other files: nodes `x` and `y`, and a triangle-only `element` array. It had no UGRID mesh variable, so `SCHISM_hgrid` is created from `x`, `y` and `element` (no edges). `Conventions`, `title` and `history` are added, and the `coordinates` attribute of every node variable is rewritten as `x y`.

| Variable | Changes |
| --- | --- |
| `element` | Invalid `standard_name = face_node_connectivity` and units `nondimensional` removed; `cf_role = face_node_connectivity`, `start_index = 1` and a `long_name` added |
| `x`, `y` | `long_name` set to node longitude and node latitude; the `positive` attribute removed (it is only for vertical coordinates) |
| `depth` | Invalid `standard_name` replaced by `sea_floor_depth_below_geoid`; `positive = down`; its `coordinates` attribute, which listed `time` although `depth` has no time dimension, becomes `x y` |
| `zeta` | Invalid `standard_name` replaced by `sea_surface_height_above_geoid`; units `m` |
| `temp_surface`, `temp_bottom` | `sea_surface_temperature`, `sea_water_temperature_at_sea_floor`; units `deg C` changed to `degree_Celsius`; `units_metadata = "temperature: on_scale"` |
| `salt_surface`, `salt_bottom` | Units `psu` changed to `1`; no standard name |
| `uvel_*`, `vvel_*` (surface, bottom, 0.5 m, 4.5 m) | Units `m/s` changed to `m s-1`; no standard name |
| `uvel4.5`, `vvel4.5`, `uvel0.5`, `vvel0.5` | Optionally renamed to `uvel_4p5m`, `vvel_4p5m`, `uvel_0p5m`, `vvel_0p5m` (CF recommends letters, digits and underscores only). This is the only variable renaming; set `RENAME_DOTTED = False` in the fix cell to keep the original names |

In the test cell, keep the `RENAMES` dictionary if you renamed the four variables.


In [16]:
from datetime import datetime, timezone

import numpy as np
import xarray as xr

# ---- set your files here ---------------------------------------------------
src = "stofs_3d_atl.t12z.field2d_f025_036.nc"
out = "stofs_3d_atl.t12z.field2d_f025_036_compliant.nc"

# CF 2.3 recommends names made of letters, digits and underscores. Four
# variables contain a dot. True renames them (downstream code that reads the
# old names must be updated); False keeps the names and leaves 4 CF warnings.
RENAME_DOTTED = True
# ----------------------------------------------------------------------------

# decode_coords=False keeps the raw "coordinates" attributes, so they can be fixed
ds = xr.open_dataset(src, mask_and_scale=False, decode_times=False, decode_coords=False)

# ===== Optional: rename variables with dots ==================================
if RENAME_DOTTED:
    renames = {"uvel4.5": "uvel_4p5m", "vvel4.5": "vvel_4p5m",
               "uvel0.5": "uvel_0p5m", "vvel0.5": "vvel_0p5m"}
    ds = ds.rename({k: v for k, v in renames.items() if k in ds.variables})

# ===== UGRID: this file has NO mesh topology variable, so create one =========
# Triangular mesh: nodes (x, y) and face-node connectivity (element). The file
# has no edge array, which is optional in UGRID.
ds["SCHISM_hgrid"] = xr.Variable((), np.int32(0), {
    "cf_role": "mesh_topology",
    "long_name": "Topology data of 2d unstructured mesh",
    "topology_dimension": np.int32(2),
    "node_coordinates": "x y",
    "face_node_connectivity": "element",
})

# connectivity: keep the name "element"; remove the invalid standard_name and
# the units (a connectivity array has none)
el = ds["element"]
el.attrs.pop("standard_name", None)
el.attrs.pop("units", None)
el.attrs.update(cf_role="face_node_connectivity",
                long_name="face-node connectivity (triangular elements)",
                start_index=np.int32(1))

# ===== CF: coordinates ========================================================
ds["x"].attrs.update(standard_name="longitude", long_name="node longitude", units="degrees_east")
ds["y"].attrs.update(standard_name="latitude", long_name="node latitude", units="degrees_north")
for v in ("x", "y"):
    ds[v].attrs.pop("positive", None)   # "positive" is only for vertical coordinates

# ===== CF: names and units (check against your data) =========================
META = {
    "depth": dict(standard_name="sea_floor_depth_below_geoid", units="m", positive="down"),
    "zeta": dict(standard_name="sea_surface_height_above_geoid",
                 long_name="water surface elevation above xgeoid20b", units="m"),
    "temp_surface": dict(standard_name="sea_surface_temperature",
                         units="degree_Celsius", units_metadata="temperature: on_scale"),
    "temp_bottom": dict(standard_name="sea_water_temperature_at_sea_floor",
                        units="degree_Celsius", units_metadata="temperature: on_scale"),
    # "psu" is not a UDUNITS unit; practical salinity is dimensionless ("1")
    "salt_surface": dict(units="1"),
    "salt_bottom": dict(units="1"),
}
VEL_NAMES = ["uvel_surface", "uvel_bottom", "vvel_surface", "vvel_bottom"]
VEL_NAMES += (["uvel_4p5m", "vvel_4p5m", "uvel_0p5m", "vvel_0p5m"] if RENAME_DOTTED
              else ["uvel4.5", "vvel4.5", "uvel0.5", "vvel0.5"])
for v in VEL_NAMES:
    META[v] = dict(units="m s-1")   # normalizes "m/s"
for name, attrs in META.items():
    if name in ds.variables:
        ds[name].attrs.update(attrs)

# ===== CF section 5: "coordinates" attributes =================================
# The checker reported "time" as an auxiliary coordinate of "depth" (which has no
# time dimension). Rewrite the attribute: node variables point at x and y only.
for name in ds.data_vars:
    if name in ("SCHISM_hgrid", "element", "x", "y"):
        continue
    if "node" in ds[name].dims:
        ds[name].attrs["coordinates"] = "x y"
        ds[name].attrs["mesh"] = "SCHISM_hgrid"
        ds[name].attrs["location"] = "node"
    else:
        ds[name].attrs.pop("coordinates", None)
ds["element"].attrs.pop("coordinates", None)

# CF: time calendar stated explicitly (it is the default)
ds["time"].attrs["calendar"] = "standard"

# ===== CF: global attributes =================================================
ds.attrs["Conventions"] = "CF-1.11 UGRID-1.0"
stamp = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
ds.attrs["history"] = (f"{stamp}: metadata made CF-1.11/UGRID-1.0 compliant "
                       "(data values unchanged)")

ds.to_netcdf(out, encoding={
    "time": {"_FillValue": None},
    "x": {"_FillValue": None},
    "y": {"_FillValue": None},
})
print("Wrote", out)

Wrote stofs_3d_atl.t12z.field2d_f025_036_compliant.nc


**Test.** Compares the new file with the original: values, statistics, fill handling, connectivity, and a read through xugrid.


In [17]:
"""
Check that a *_compliant.nc file still holds the same data as the original:

 1. every numeric variable has identical values (NaN positions included)
 2. statistics (valid count, masked count, sum, mean, min, max) are identical
 3. fill / missing values are understood: the number of masked (NaN) values in
    the decoded new file equals the number of raw _FillValue (+ NaN) values
 4. the connectivity arrays are unchanged
 5. xugrid opens the new file, builds the mesh with the right sizes and
    connectivity, and a xugrid reduction (mean) matches xarray on the original
"""
import sys

import numpy as np
import xarray as xr
import xugrid as xu

# ---- set your files here ---------------------------------------------------
ORIG = "stofs_3d_atl.t12z.field2d_f025_036.nc"
NEW = "stofs_3d_atl.t12z.field2d_f025_036_compliant.nc"

# variables that were renamed in the new file (field2d), else leave empty:
RENAMES = {"uvel4.5": "uvel_4p5m", "vvel4.5": "vvel_4p5m",
            "uvel0.5": "uvel_0p5m", "vvel0.5": "vvel_0p5m"}
# RENAMES = {}

# which time steps to test: None = all (slow on 8 GB files), or e.g. [0, -1]
TIME_STEPS = None
# ----------------------------------------------------------------------------

SKIP = {"SCHISM_hgrid"}   # placeholder variable; its type was changed on purpose
FACE_CANDIDATES = ["SCHISM_hgrid_face_nodes", "element"]
EDGE_CANDIDATES = ["SCHISM_hgrid_edge_nodes"]

results = []


def check(name, ok, msg=""):
    results.append((name, bool(ok)))
    print(("PASS  " if ok else "FAIL  ") + name + (f"   {msg}" if msg else ""))


class Stats:
    def __init__(self):
        self.n = 0          # valid values
        self.nnan = 0       # masked (NaN) values
        self.sum = 0.0
        self.min = np.inf
        self.max = -np.inf

    def add(self, a):
        if a.dtype.kind == "f":
            nan = int(np.isnan(a).sum())
            self.nnan += nan
            valid = a.size - nan
            if valid:
                self.n += valid
                self.sum += float(np.nansum(a, dtype=np.float64))
                self.min = min(self.min, float(np.nanmin(a)))
                self.max = max(self.max, float(np.nanmax(a)))
        else:
            self.n += a.size
            self.sum += float(a.sum(dtype=np.float64))
            self.min = min(self.min, float(a.min()))
            self.max = max(self.max, float(a.max()))

    @property
    def mean(self):
        return self.sum / self.n if self.n else float("nan")

    def same_as(self, o):
        return (self.n, self.nnan, self.sum, self.min, self.max) == \
               (o.n, o.nnan, o.sum, o.min, o.max)


def step_indices(da):
    """Time steps to test for a variable (None = the whole variable at once)."""
    if "time" in da.dims and da.ndim > 1:
        n = da.sizes["time"]
        return list(range(n)) if TIME_STEPS is None else sorted({i % n for i in TIME_STEPS})
    return [None]


def get(da, i):
    return da.values if i is None else da.isel(time=i).values


def as_xr(x):
    return getattr(x, "obj", x)   # unwrap xugrid objects


# ---- open everything --------------------------------------------------------
orig_dec = xr.open_dataset(ORIG, decode_times=False)                       # fills masked to NaN
new_dec = xr.open_dataset(NEW, decode_times=False)
orig_raw = xr.open_dataset(ORIG, mask_and_scale=False, decode_times=False)  # as stored
new_raw = xr.open_dataset(NEW, mask_and_scale=False, decode_times=False)
uds = xu.open_dataset(NEW)

face_name = next(n for n in FACE_CANDIDATES if n in orig_raw.variables)
edge_name = next((n for n in EDGE_CANDIDATES if n in orig_raw.variables), None)
conn_names = {face_name} | ({edge_name} if edge_name else set())

# ---- 1-3. every numeric variable --------------------------------------------
print("\n=== Variables: values, statistics, fill/missing handling ===")
rows = []
for v in orig_dec.variables:
    if v in SKIP or v in conn_names:
        continue
    od = orig_dec[v]
    if od.dtype.kind not in "fiu":
        continue
    rn = RENAMES.get(v, v)
    if rn not in new_dec.variables:
        check(v, False, "missing in new file")
        continue
    nd, nr = new_dec[rn], new_raw[rn]
    if od.dims != nd.dims or od.shape != nd.shape:
        check(v, False, f"dims/shape differ: {od.dims}{od.shape} vs {nd.dims}{nd.shape}")
        continue

    so, sn = Stats(), Stats()
    identical = True
    n_fill_raw = n_nan_raw = 0
    fill = nr.attrs.get("_FillValue")
    for i in step_indices(od):
        a_o, a_n, raw = get(od, i), get(nd, i), get(nr, i)
        identical &= np.array_equal(a_o, a_n, equal_nan=(a_o.dtype.kind == "f"))
        so.add(a_o)
        sn.add(a_n)
        if fill is not None:
            n_fill_raw += int((raw == fill).sum())
        if raw.dtype.kind == "f":
            n_nan_raw += int(np.isnan(raw).sum())

    ok_vals = identical and so.same_as(sn)
    expected_masked = n_fill_raw + n_nan_raw
    ok_fill = sn.nnan == expected_masked
    ok_attr = ("missing_value" not in nr.attrs) or (nr.attrs.get("missing_value") == fill)
    check(f"{v}: values and statistics identical", ok_vals,
          "" if ok_vals else f"orig n={so.n} sum={so.sum} | new n={sn.n} sum={sn.sum}")
    check(f"{v}: fill/missing understood", ok_fill and ok_attr,
          f"masked in decoded new = {sn.nnan}, stored fill/NaN = {expected_masked}, "
          f"_FillValue = {fill}")

    # xugrid reading of the same variable (first tested time step)
    if rn != "time" and rn not in as_xr(uds).variables:
        # xugrid does not expose every variable (e.g. the "crs" grid-mapping
        # variable); the xarray checks above already cover it
        print(f"NOTE  {v}: not exposed by xugrid, xugrid comparison skipped")
    elif rn != "time":
        try:
            i0 = step_indices(od)[0]
            ua = uds[rn]
            xa = as_xr(ua)
            a_u = get(xa, i0)
            res_u = ua if i0 is None else ua.isel(time=i0)
            mean_u = float(as_xr(res_u.mean()))
            od0 = od if i0 is None else od.isel(time=i0)
            mean_o = float(od0.mean())
            ok_u = np.array_equal(a_u, get(od, i0), equal_nan=(a_u.dtype.kind == "f")) \
                and np.isclose(mean_u, mean_o, rtol=1e-6, atol=0, equal_nan=True)
            check(f"{v}: xugrid values and mean match original", ok_u,
                  f"xugrid mean = {mean_u:.6g}, original mean = {mean_o:.6g}")
        except Exception as e:  # report, don't stop
            check(f"{v}: xugrid values and mean match original", False, repr(e))

    rows.append((v, str(od.dtype), sn.n, sn.nnan, sn.mean, sn.min, sn.max))

# ---- 4. connectivity arrays (as stored) -------------------------------------
print("\n=== Connectivity arrays ===")
for name in sorted(conn_names):
    fo = np.nan_to_num(orig_raw[name].values.astype("float64"), nan=-1).astype("int64")
    fn = new_raw[name].values.astype("int64")
    check(f"{name}: unchanged as stored", np.array_equal(fo, fn))

# ---- 5. xugrid mesh ---------------------------------------------------------
print("\n=== xugrid mesh ===")
grid = uds.ugrid.grids[0]
mesh_var = next(n for n, x in new_raw.variables.items()
                if x.attrs.get("cf_role") == "mesh_topology")
node_x_name = new_raw[mesh_var].attrs["node_coordinates"].split()[0]
fn = new_raw[face_name].values.astype("int64")
start = int(new_raw[face_name].attrs.get("start_index", 0))
check("mesh variable present", True, f"'{mesh_var}'")
check("n_node matches", grid.n_node == new_raw[node_x_name].size,
      f"xugrid {grid.n_node}, file {new_raw[node_x_name].size}")
check("n_face matches", grid.n_face == fn.shape[0],
      f"xugrid {grid.n_face}, file {fn.shape[0]}")
expected = np.where(fn >= start, fn - start, -1)   # xugrid uses 0-based indices
gc = np.asarray(grid.face_node_connectivity)
check("face connectivity read correctly by xugrid",
      gc.shape == expected.shape and np.array_equal(gc, expected),
      f"shape {gc.shape} vs expected {expected.shape}")

# ---- summary ----------------------------------------------------------------
print("\n=== Statistics of the new file (decoded; fill values excluded) ===")
print(f"{'variable':28s}{'dtype':9s}{'valid':>14s}{'masked':>12s}{'mean':>14s}{'min':>14s}{'max':>14s}")
for r in rows:
    print(f"{r[0]:28s}{r[1]:9s}{r[2]:14d}{r[3]:12d}{r[4]:14.6g}{r[5]:14.6g}{r[6]:14.6g}")

failed = [n for n, ok in results if not ok]
print(f"\n{len(results) - len(failed)} passed, {len(failed)} failed")
for n in failed:
    print("  FAILED:", n)
sys.exit(1 if failed else 0)


=== Variables: values, statistics, fill/missing handling ===
PASS  depth: values and statistics identical
PASS  depth: fill/missing understood   masked in decoded new = 0, stored fill/NaN = 0, _FillValue = nan
PASS  depth: xugrid values and mean match original   xugrid mean = 484.262, original mean = 484.262
PASS  zeta: values and statistics identical
PASS  zeta: fill/missing understood   masked in decoded new = 14737980, stored fill/NaN = 14737980, _FillValue = -99999.0
PASS  zeta: xugrid values and mean match original   xugrid mean = 0.534736, original mean = 0.534736
PASS  temp_surface: values and statistics identical
PASS  temp_surface: fill/missing understood   masked in decoded new = 14737980, stored fill/NaN = 14737980, _FillValue = -99999.0
PASS  temp_surface: xugrid values and mean match original   xugrid mean = 23.4041, original mean = 23.4041
PASS  temp_bottom: values and statistics identical
PASS  temp_bottom: fill/missing understood   masked in decoded new = 14737980, sto

SystemExit: 0

/lustre/env/miniconda3/envs/STOFS-Observer/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
